# A10: Data Understanding on the unified dataset (Tanner Edwards)

Personal notebook. Re-runs Matt's Data Understanding (`Sorensen-A10.ipynb` §2–§3, plan steps 8–9) on the **unified dataset** built by `00_build_dataset.ipynb`, which adds four sources and grows the usable population from 5,766 to about 6,800 films. Same helpers, same review flags, same privacy rule.

**Run `00_build_dataset.ipynb` first.** This notebook only reads `data/clean/films_unified.csv` and the raw files; it writes nothing.

| Section | What | Matt's equivalent |
|---|---|---|
| 1 | Setup and Matt's profile helpers | §1, §3 helpers |
| 2 | Data Receipt for the **new** sources | §2 |
| 3a | Field inventory: all columns, role at the financing decision | §3a |
| 3b | The label on the expanded population | §3b |
| 3c–3f | Numeric, categorical, identifier/text and date fields | §3c–§3h |
| 3g | **New:** do the sources agree with each other? | — |
| 3h | **New:** who has which data (coverage by period and budget tier) | §4a idea |
| 3i | Flag register | §3i |
| 3j–3k | Findings and proposed dispositions (need Tanner + Matt sign-off) | §3j–§3k |

This section **observes and records; it doesn't clean.**

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
YEARS = (2000, 2023)

films = pd.read_csv(CLEAN / "films_unified.csv", low_memory=False)
for c in ["release_dt", "tn_Release Date", "bom_first_date", "sl_original_release_date",
          "rt_releaseDateTheaters", "rt_releaseDateStreaming", "rtrev_first_review", "slrev_first_review"]:
    films[c] = pd.to_datetime(films[c], errors="coerce")
print(f"films_unified.csv: {len(films):,} films x {films.shape[1]} columns; {films['usable'].sum():,} usable")

Matt's chart style and profile helpers, copied unchanged from `Sorensen-A10.ipynb` §3 so the two profiles are directly comparable.

In [ ]:
SERIES = {"train 2000-15": "#2a78d6", "test 2016-19": "#eb6834", "stress 2020-23": "#1baf7a"}
BLUE, BLUE_LIGHT, INK, INK2 = "#2a78d6", "#cde2fb", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ad", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "axes.titlesize": 10, "axes.titlelocation": "left", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
})

SKEW_FLAG, RARE_FLAG = 0.5, 0.05   # Ch 6 review flags
FLAGS = []

def flag(file, field, issue, evidence):
    FLAGS.append({"file": file, "field": field, "issue": issue, "evidence": evidence})


def numeric_profile(df, cols, zero_as_missing=()):
    rows = []
    for c in cols:
        s = df[c].astype(float)
        if c in zero_as_missing:
            s = s.mask(s == 0)
        x = s.dropna()
        q1, q3 = x.quantile([0.25, 0.75])
        iqr = q3 - q1
        rows.append({
            "field": c, "n": len(x), "% missing": round(100 * s.isna().mean(), 1),
            "mean": x.mean(), "median": x.median(), "std": x.std(),
            "min": x.min(), "Q1": q1, "Q3": q3, "max": x.max(), "IQR": iqr, "skew": x.skew(),
            "# above Q3+1.5 IQR": int((x > q3 + 1.5 * iqr).sum()),
            "# below Q1-1.5 IQR": int((x < q1 - 1.5 * iqr).sum()),
        })
    out = pd.DataFrame(rows).set_index("field")
    out["skew flag"] = out["skew"].abs() > SKEW_FLAG
    return out


def category_profile(s, name, multi=False, names_hidden=False):
    n = s.notna().sum()
    vals = s.dropna().astype(str)
    if multi:
        vals = vals.str.split(",").explode().str.strip()
        vals = vals[vals.ne("")]
    vc = vals.value_counts()
    share = vc / n
    row = {
        "field": name, "% missing": round(100 * s.isna().mean(), 1), "multi-label": multi,
        "levels": len(vc),
        "top level": "(person, withheld)" if names_hidden else (vc.index[0] if len(vc) else None),
        "top level % of rows": round(100 * share.iloc[0], 1) if len(vc) else None,
        "levels < 5% of rows": int((share < RARE_FLAG).sum()),
        "% of rows in a level < 5%": None if multi else round(100 * share[share < RARE_FLAG].sum(), 1),
    }
    return row, vc


def id_text_profile(df, cols):
    rows = []
    for c in cols:
        s = df[c]
        nn = s.dropna().astype(str)
        rows.append({
            "field": c, "% missing": round(100 * s.isna().mean(), 1),
            "unique values": nn.nunique(),
            "rows sharing a value": int(nn.duplicated(keep=False).sum()),
            "median length (chars)": nn.str.len().median(),
        })
    return pd.DataFrame(rows).set_index("field")


def box_hist(ax_box, ax_hist, x, title, xlabel, ylabel, log=False):
    x = x.dropna().astype(float)
    if log and (x <= 0).any():
        title += f"\n({(x <= 0).sum():,} values <= 0 not shown on the log scale)"
        x = x[x > 0]
    bins = int(np.ceil(len(x) ** (1 / 3)))
    edges = np.logspace(np.log10(x.min()), np.log10(x.max()), bins + 1) if log else bins
    ax_hist.hist(x, bins=edges, color=BLUE, edgecolor="white", linewidth=0.4)
    ax_box.boxplot(x, orientation="horizontal", widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor=BLUE_LIGHT, edgecolor=BLUE),
                   medianprops=dict(color=INK, linewidth=1.5),
                   whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE),
                   flierprops=dict(marker="o", markersize=2.5, markerfacecolor=BLUE,
                                   markeredgecolor="none", alpha=0.35))
    ax_hist.axvline(x.median(), color=INK, lw=1, label=f"median {x.median():,.3g}")
    ax_hist.axvline(x.mean(), color=INK, lw=1, ls="--", label=f"mean {x.mean():,.3g}")
    if log:
        ax_hist.set_xscale("log")
    ax_box.set_yticks([])
    ax_box.grid(False)
    ax_box.set_title(title)
    ax_hist.set_xlabel(xlabel)
    ax_hist.set_ylabel(ylabel)
    ax_hist.legend(fontsize=7, frameon=False)


def profile_grid(specs, ylabel, ncols=3):
    nrows = int(np.ceil(len(specs) / ncols))
    fig = plt.figure(figsize=(4.3 * ncols, 3.1 * nrows), layout="constrained")
    subfigs = np.atleast_1d(fig.subfigures(nrows, ncols)).ravel()
    for sf, (x, title, xlabel, log) in zip(subfigs, specs):
        ax_box, ax_hist = sf.subplots(2, 1, sharex=True, height_ratios=[1, 4])
        box_hist(ax_box, ax_hist, x, title, xlabel, ylabel, log)
    for sf in subfigs[len(specs):]:
        sf.set_visible(False)
    plt.show()


def count_plot(ax, vc, n_rows, title, top=10, xlabel="% of rows"):
    shares = (100 * vc / n_rows).head(top)[::-1]
    ax.barh(shares.index.astype(str).str.slice(0, 35), shares.values, color=BLUE, height=0.7)
    ax.axvline(100 * RARE_FLAG, color=INK2, ls=":", lw=1)
    for y, v in enumerate(shares.values):
        ax.text(v, y, f" {v:.1f}%", va="center", fontsize=7, color=INK2)
    ax.set_title(title + (f"  (top {top} of {len(vc):,})" if len(vc) > top else ""))
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)

print("Profile helpers defined.")

## 2. Data Receipt for the new sources (plan step 8)

Matt's §2 covers TMDB, RT movies (andrezaza) and RT reviews. These are the sources added since. Licenses were read from the Kaggle API (`licenseName`) and the GitHub API on 2026-11-23.

| Field | The Numbers | Box Office Mojo | Wikidata snapshot | RT movies (stefanoleone992) | RT critic reviews (stefanoleone992) |
|---|---|---|---|---|---|
| **File** | `the_numbers/Top Movies (Cleaned Data).csv` | `boxofficemojo/revenues_per_day.csv.gz` | `wikidata/wikidata_film_financials_2026-11-21.csv` | `stefanoleone992/rotten_tomatoes_movies.csv` | `stefanoleone992/rotten_tomatoes_critic_reviews.csv` |
| **Publisher** | michaelmatta0 (Kaggle) | tjwaterman99 (GitHub release) | Our own SPARQL query (`tools/wikidata_films.rq`) | stefanoleone992 (Kaggle) | same |
| **Original source** | the-numbers.com (Nash Information Services) | boxofficemojo.com (IMDb) | Wikidata (Wikimedia, crowd-edited) | Rotten Tomatoes | Rotten Tomatoes |
| **URL** | kaggle.com/datasets/michaelmatta0/movies-ultimate-metrics-features-and-metadata | github.com/tjwaterman99/boxofficemojo-scraper | query.wikidata.org | kaggle.com/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset | same |
| **Access date** | 2026-11-21 | 2026-11-20 | 2026-11-21 (snapshot committed to git) | 2026-11-20 | 2026-11-20 |
| **License (as stated)** | MIT | **None stated** | CC0 1.0 | CC0 1.0 | CC0 1.0 |
| **Claimed grain** | One row per film | One row per film **per day** | One row per film with a USD budget and a TMDB id | One row per RT film (link) | One row per critic review |
| **Population** | ~6.6k "top" films by budget, all years | Films on the US daily chart, 2000–2025 | Wikidata films 2000–2023 with a USD cost | ~17.7k RT films (2020 scrape) | Reviews of those films |
| **Sensitive fields** | None | None | None | `directors`, `authors`, `actors`: public professionals | `critic_name`, `review_content`: named people and copyrighted text |
| **Rights caveats** | Scraped; The Numbers' terms aren't overridden by the uploader's MIT | Scraped; no license, so academic fair use only | — | Scores belong to RT | Review text is copyrighted; not copied into the unified file |
| **Permitted use here** | Academic, non-commercial. Cite the-numbers.com. | Same, and cite Box Office Mojo | Cite Wikidata | Same as Matt's RT files | Same; counts only |

**Also note:** `get_data.py` downloads `rotten_tomatoes_movie_reviews.csv` from **priyamchoksi** (license MIT), while Matt's receipt cites the **andrezaza** upload (CC0). Same row count (1,444,963). The receipt should cite one; open question for Matt.

### 2a. File fingerprints
SHA-256 for every raw file, Matt's originals included, so all of them can be checked as unchanged at submission.

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

FILES = sorted(p for p in RAW.rglob("*") if p.is_file() and p.name != ".gitkeep")
fingerprints = pd.DataFrame([{"file": str(p.relative_to(RAW)), "MB": round(p.stat().st_size / 1e6, 1),
                              "sha256": sha256(p)} for p in FILES])
with pd.option_context("display.max_colwidth", 70):
    display(fingerprints)

### 2b. Grain check (new sources)
Does one row equal what the source claims? Box Office Mojo's `id` turned out to be per row, not per film (it was collapsed to one row per title in the build).

In [ ]:
tn_raw = pd.read_csv(RAW / "the_numbers/Top Movies (Cleaned Data).csv", low_memory=False)
bom_raw = pd.read_csv(RAW / "boxofficemojo/revenues_per_day.csv.gz")
wd_raw = pd.read_csv(RAW / "wikidata/wikidata_film_financials_2026-11-21.csv")
sl_raw = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_movies.csv", low_memory=False)
slr_raw = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_critic_reviews.csv", low_memory=False,
                      usecols=["rotten_tomatoes_link", "critic_name", "review_date", "review_type", "top_critic"])

def grain_check(df, key):
    return {"rows": len(df), "key": key, "unique keys": df[key].nunique(),
            "rows with a repeated key": int(df[key].duplicated(keep=False).sum()),
            "exact duplicate rows": int(df.duplicated().sum())}

grain = pd.DataFrame({
    "The Numbers (Movie URL)": grain_check(tn_raw, "Movie URL"),
    "Box Office Mojo (id)": grain_check(bom_raw, "id"),
    "Box Office Mojo (title + date)": grain_check(bom_raw.assign(k=bom_raw["title"] + "|" + bom_raw["date"]), "k"),
    "Wikidata (tmdb)": grain_check(wd_raw, "tmdb"),
    "RT sl movies (link)": grain_check(sl_raw, "rotten_tomatoes_link"),
    "RT sl reviews (link + critic + date)": grain_check(
        slr_raw.assign(k=slr_raw["rotten_tomatoes_link"] + "|" + slr_raw["critic_name"].fillna("?") + "|" + slr_raw["review_date"].astype(str)), "k"),
})
grain

### 2c. Date coverage and placeholder dates (new sources)

In [ ]:
dates = {
    "The Numbers Release Date": pd.to_datetime(tn_raw["Release Date"], errors="coerce"),
    "Box Office Mojo date": pd.to_datetime(bom_raw["date"], errors="coerce"),
    "RT sl original_release_date": pd.to_datetime(sl_raw["original_release_date"], errors="coerce"),
    "RT sl review_date": pd.to_datetime(slr_raw["review_date"], errors="coerce"),
}
ACCESS = pd.Timestamp("2026-11-21")
pd.DataFrame({n: {"min": d.min().date(), "max": d.max().date(), "% missing": round(100 * d.isna().mean(), 1),
                  "before 1900": int((d.dt.year < 1900).sum()), "after 2023": int((d.dt.year > 2023).sum()),
                  "after access date": int((d > ACCESS).sum())} for n, d in dates.items()}).T

### 2d. Population: where the usable films come from

Same funnel as Matt's §2f, but on the unified table, with the source of each film's budget and revenue. The label period split is the contract's.

In [ ]:
f = films.copy()
f["period"] = pd.cut(f["year"], [1999, 2015, 2019, 2023], labels=list(SERIES))
steps = [
    ("2000-2023 film with any budget or revenue (unified table)", pd.Series(True, index=f.index)),
    ("budget >= $1M (any source)", f["fin_budget"].ge(1_000_000)),
    ("revenue > 0 (any source)", f["fin_revenue"].gt(0)),
]
mask = pd.Series(True, index=f.index)
funnel = []
for label, cond in steps:
    mask &= cond
    funnel.append({"filter (cumulative)": label, "films remaining": int(mask.sum())})
display(pd.DataFrame(funnel))
assert mask.sum() == f["usable"].sum()

u = f[f["usable"]].copy()
display(pd.crosstab(u["fin_budget_src"], u["fin_revenue_src"], margins=True).rename_axis(index="budget from", columns="revenue from"))
display(pd.crosstab(u["period"], u["in_tmdb"].map({True: "TMDB film", False: "The Numbers only"}), margins=True))

# S4 restated on the unified table: budget >= $1M but no revenue from any source
b1 = f["fin_budget"].ge(1_000_000)
s4 = pd.DataFrame({p: {"budget >= $1M": int((b1 & (f["period"] == p)).sum()),
                       "with revenue": int((b1 & f["fin_revenue"].gt(0) & (f["period"] == p)).sum())}
                   for p in SERIES}).T
s4["% missing revenue"] = (100 * (1 - s4["with revenue"] / s4["budget >= $1M"])).round(1)
display(s4)
flag("Unified", "fin_revenue", "missing (S4)",
     f"{100 * (1 - (b1 & f['fin_revenue'].gt(0)).sum() / b1.sum()):.1f}% of budget >= $1M films have no revenue "
     f"from any source (Matt's TMDB-only figure: 43.6%)")

## 3. Univariate profile of every field (plan step 9)

**Which rows:** the usable films (2000–2023, budget ≥ $1M, revenue > 0 from any source), the population the screen will score. The label is shown by period. **Review flags (Ch 6):** |skew| > 0.5; any category level held by < 5% of non-missing rows. **Privacy:** director, writer, actor and critic names are profiled by counts only.

In [ ]:
pop = u
pop["multiple"] = pop["fin_revenue"] / pop["fin_budget"]
pop["log10_multiple"] = np.log10(pop["multiple"])
for k in [1, 2, 2.5, 3]:
    pop[f"loser_{k}x"] = pop["multiple"] < k
pop["money_loser"] = pop["loser_2.5x"]
pop["release_month"] = pop["release_dt"].dt.month
pop["n_genres"] = pop["genres"].str.split(",").str.len()
print(f"Population: {len(pop):,} films")
print(pop["period"].value_counts().sort_index().to_string())

### 3a. Field inventory
Every column in the unified file, with its source and what the contract lets it do at the financing decision.
- **allowed input**: on the contract's input list (budget, genre, director/company track record, runtime, release month/quarter, language, country)
- **candidate input (needs amendment)**: known before production, but not on the contract's list (e.g. MPAA rating, source material, franchise, writer, cast). Using it needs a dated amendment.
- **prohibited (post-release)**, **outcome part**, **key / join**, **build metadata**, **not used**

In [ ]:
A, CAND, P, K, O, B, N = ("allowed input", "candidate input (needs amendment)", "prohibited (post-release)",
                         "key / join", "outcome part (post-release)", "build metadata", "not used")

def source_of(c):
    for pre, name in [("tn_", "The Numbers"), ("bom_", "Box Office Mojo"), ("sl_", "RT sl movies"),
                      ("wd_", "Wikidata"), ("rtrev_", "RT reviews (rollup)"), ("slrev_", "RT sl reviews (rollup)"),
                      ("rt_", "RT movies"), ("fin_", "Combined")]:
        if c.startswith(pre):
            return name
    return "Build" if c in ("film_uid", "usable", "in_tmdb", "merged_ids", "release_dt", "year") else "TMDB"

ROLE = {
    # TMDB
    "film_uid": K, "id": K, "title": K, "original_title": K, "imdb_id": K, "release_date": A, "release_dt": A, "year": A,
    "budget": A, "revenue": O, "runtime": A, "genres": A, "production_companies": A, "production_countries": A,
    "original_language": A, "spoken_languages": N, "keywords": P, "vote_average": P, "vote_count": P,
    "popularity": P, "status": B, "adult": B, "overview": N, "tagline": N, "homepage": N, "poster_path": N,
    "backdrop_path": N,
    # combined / build
    "fin_budget": A, "fin_revenue": O, "fin_budget_src": B, "fin_revenue_src": B, "usable": B, "in_tmdb": B,
    "merged_ids": B,
    # The Numbers
    "tn_Movie Name": K, "tn_Release Date": A, "tn_Production Budget (USD)": A, "tn_Running Time (minutes)": A,
    "tn_Genre": A, "tn_Production/Financing Companies": A, "tn_Production Countries": A, "tn_Languages": A,
    "tn_MPAA Rating": CAND, "tn_Source": CAND, "tn_Creative Type": CAND, "tn_Production Method": CAND,
    "tn_Franchise": CAND, "tn_Keywords": P, "tn_Movie URL": K, "tn_year_diff": B,
    "tn_Domestic Releases": P, "tn_International Releases": P, "tn_Theater counts": P, "tn_Video Release": P,
    "tn_Legs": P, "tn_Opening Weekend (USD)": P, "tn_Domestic Share Percentage": P,
    "tn_Est. Domestic DVD Sales (USD)": P, "tn_Est. Domestic Blu-ray Sales (USD)": P,
    "tn_Total Est. Domestic Video Sales (USD)": P,
    # Box Office Mojo
    "bom_title": K, "bom_year_diff": B, "bom_first_date": P, "bom_distributor": CAND,
    # RT sl movies
    "sl_rotten_tomatoes_link": K, "sl_movie_title": K, "sl_year_diff": B, "sl_genres": A, "sl_runtime": A,
    "sl_production_company": A, "sl_directors": A, "sl_authors": CAND, "sl_actors": CAND,
    "sl_content_rating": CAND, "sl_original_release_date": A, "sl_movie_info": N, "sl_critics_consensus": P,
    "sl_streaming_release_date": P,
    # RT movies (andrezaza)
    "rt_id": K, "rt_title": K, "rt_matched_on": B, "rt_rule": B, "rt_year_diff": B, "rt_runtime_diff": B,
    "rt_director": A, "rt_writer": CAND, "rt_rating": CAND, "rt_ratingContents": CAND, "rt_genre": A,
    "rt_originalLanguage": A, "rt_runtimeMinutes": A, "rt_releaseDateTheaters": A,
    "rt_releaseDateStreaming": P, "rt_distributor": CAND, "rt_soundMix": N, "rt_boxOffice": O,
    "rt_boxOffice_usd": O,
    # Wikidata
    "wd_item": K, "wd_title": K, "wd_year": A, "wd_wd_budget_usd": A, "wd_wd_box_office_usd": O,
}
def role_of(c):
    if c in ROLE:
        return ROLE[c]
    if c.startswith(("tn_Domestic", "tn_International", "tn_Worldwide", "tn_Infl", "bom_domestic", "bom_days", "bom_max")):
        return O if "Gross" in c or "Box Office" in c or "gross" in c else P
    if c.startswith(("sl_tomatometer", "sl_audience", "rt_audience", "rt_tomato", "rtrev_", "slrev_")):
        return P
    return None

inventory = pd.DataFrame({"field": films.columns})
inventory["source"] = inventory["field"].map(source_of)
inventory["dtype"] = inventory["field"].map(lambda c: str(films[c].dtype))
inventory["% filled (usable films)"] = inventory["field"].map(lambda c: round(100 * pop[c].notna().mean(), 1))
inventory["role at financing decision"] = inventory["field"].map(role_of)
assert inventory["role at financing decision"].notna().all(), inventory.loc[inventory["role at financing decision"].isna(), "field"].tolist()
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 60):
    display(inventory)
pd.crosstab(inventory["source"], inventory["role at financing decision"], margins=True)

### 3b. The label on the expanded population
Money-loser = combined revenue < 2.5 × combined budget, with 1×, 2×, 3× sensitivity. Compared with Matt's TMDB-only figures (64.6% losers overall), and split by where the financials came from.

In [ ]:
def label_summary(g):
    return pd.Series({
        "films": len(g), "median multiple": g["multiple"].median(),
        "Q1": g["multiple"].quantile(0.25), "Q3": g["multiple"].quantile(0.75),
        "skew (log10 multiple)": g["log10_multiple"].skew(),
        **{f"% losers at {k}x": 100 * g[f"loser_{k}x"].mean() for k in [1, 2, 2.5, 3]},
    })

label_tbl = pd.DataFrame({p: label_summary(g) for p, g in pop.groupby("period", observed=True)}
                         | {"all 2000-23": label_summary(pop)}).T
display(label_tbl.round(2))

pop["fin_source"] = np.where(pop["fin_budget_src"].eq("tmdb") & pop["fin_revenue_src"].eq("tmdb"),
                             "TMDB only (Matt's population)", "uses The Numbers / Wikidata")
display(pd.DataFrame({k: label_summary(g) for k, g in pop.groupby("fin_source")}).T.round(2))

extremes = pd.Series({
    "multiple < 0.01": (pop["multiple"] < 0.01).sum(), "multiple < 0.1": (pop["multiple"] < 0.1).sum(),
    "multiple > 20": (pop["multiple"] > 20).sum(), "multiple > 100": (pop["multiple"] > 100).sum(),
}, name="films")
display(extremes.to_frame())
cols = ["title", "year", "fin_budget", "fin_revenue", "fin_budget_src", "fin_revenue_src", "multiple"]
print("Lowest 5 multiples:"); display(pop.nsmallest(5, "multiple")[cols])
print("Highest 5 multiples:"); display(pop.nlargest(5, "multiple")[cols])
loser_rate = label_tbl.loc["all 2000-23", "% losers at 2.5x"]
flag("Unified", "multiple", "extreme values",
     f"{extremes.iloc[0]} films below 0.01x and {extremes.iloc[3]} above 100x: check for unit errors")
flag("Unified", "money_loser", "majority class",
     f"{loser_rate:.1f}% losers at 2.5x: a 20% flag can catch at most {20 / loser_rate:.0%} of them")

edges = np.logspace(-3, 3, 37)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), layout="constrained", sharey=True)
for ax, (p, g) in zip(axes, pop.groupby("period", observed=True)):
    x = g["multiple"].clip(edges[0], edges[-1])
    ax.hist(x, bins=edges, weights=np.full(len(x), 100 / len(x)), color=SERIES[p], edgecolor="white", linewidth=0.4)
    ax.axvline(2.5, color=INK, lw=1, ls="--")
    ax.set_xscale("log")
    ax.set_title(f"{p}: n = {len(g):,}, {100 * g['money_loser'].mean():.0f}% losers")
    ax.set_xlabel("revenue / budget (log scale)")
axes[0].set_ylabel("% of films in period")
fig.suptitle("Label on the unified population (dashed line = 2.5x cut-off)", x=0.01, ha="left", fontsize=11)
plt.show()

### 3c. Numeric fields
Money and count fields on log scales. Runtime 0 = unknown. Post-release fields are profiled for quality only.

In [ ]:
num_cols = [c for c in films.columns if pd.api.types.is_numeric_dtype(films[c])
            and c not in ("id",) and not c.endswith("_year_diff") and c not in ("rt_runtime_diff",)
            and films[c].dtype != bool]
zero_missing = ("runtime", "tn_Running Time (minutes)", "sl_runtime", "rt_runtimeMinutes", "budget", "revenue")
prof_num = numeric_profile(pop, num_cols + ["multiple", "log10_multiple", "n_genres"], zero_as_missing=zero_missing)
prof_num.insert(0, "role", prof_num.index.map(lambda c: role_of(c) or "derived"))
with pd.option_context("display.float_format", "{:,.2f}".format, "display.max_rows", 80):
    display(prof_num)

checks = pd.Series({
    "runtime (TMDB) = 0 or missing": int(pop["runtime"].fillna(0).eq(0).sum()),
    "runtime < 60 min": int(pop["runtime"].between(1, 59).sum()),
    "runtime > 240 min": int((pop["runtime"] > 240).sum()),
    "fin_budget > $400M": int((pop["fin_budget"] > 400e6).sum()),
    "fin_budget is a round $1M multiple": int((pop["fin_budget"] % 1_000_000 == 0).sum()),
    "fin_revenue < $1,000": int((pop["fin_revenue"] < 1_000).sum()),
    "fin_revenue < $100,000": int((pop["fin_revenue"] < 100_000).sum()),
    "scores outside 0-100 (RT / sl)": int(sum((~pop[c].dropna().between(0, 100)).sum() for c in
                                              ["rt_audienceScore", "rt_tomatoMeter", "sl_tomatometer_rating", "sl_audience_rating"])),
}, name="films")
display(checks.to_frame())
flag("Unified", "fin_revenue", "implausible range",
     f"{checks['fin_revenue < $1,000']} films under $1,000; {checks['fin_revenue < $100,000']} under $100,000")
flag("Unified", "fin_budget", "estimate, not audited",
     f"{checks['fin_budget is a round $1M multiple']:,} of {len(pop):,} budgets are round $1M figures")
flag("Unified", "runtime", "implausible range",
     f"{checks['runtime (TMDB) = 0 or missing']} zero/missing, {checks['runtime < 60 min']} under 60 min, "
     f"{checks['runtime > 240 min']} over 240 min")

profile_grid([
    (pop["fin_budget"], "fin_budget (combined)", "US$ (log scale)", True),
    (pop["fin_revenue"], "fin_revenue (combined; outcome part)", "US$ (log scale)", True),
    (pop["runtime"].mask(pop["runtime"].eq(0)), "runtime (TMDB; 0 excluded)", "minutes", False),
    (pop["rt_tomatoMeter"], "rt_tomatoMeter (post-release)", "score, 0-100", False),
    (pop["rt_audienceScore"], "rt_audienceScore (post-release)", "score, 0-100", False),
    (pop["rtrev_n_reviews"], "rtrev_n_reviews (post-release)", "reviews (log scale)", True),
], ylabel="films")

### 3d. Categorical fields
Multi-label fields are split on commas, so shares add to more than 100%. Person fields (director, writer, actors) show counts only.

In [ ]:
cat_specs = [
    # (column, multi-label, names hidden)
    ("original_language", False, False), ("genres", True, False), ("production_countries", True, False),
    ("production_companies", True, False), ("spoken_languages", True, False), ("keywords", True, False),
    ("tn_MPAA Rating", False, False), ("tn_Source", False, False), ("tn_Genre", False, False),
    ("tn_Creative Type", False, False), ("tn_Production Method", True, False), ("tn_Franchise", False, False),
    ("tn_Production/Financing Companies", True, False), ("tn_Production Countries", True, False),
    ("tn_Languages", True, False), ("tn_Keywords", True, False),
    ("bom_distributor", False, False),
    ("sl_content_rating", False, False), ("sl_genres", True, False), ("sl_production_company", False, False),
    ("sl_tomatometer_status", False, False), ("sl_audience_status", False, False),
    ("rt_rating", False, False), ("rt_genre", True, False), ("rt_originalLanguage", False, False),
    ("rt_distributor", False, False), ("rt_soundMix", True, False),
    ("rt_director", True, True), ("rt_writer", True, True), ("sl_directors", True, True),
    ("sl_authors", True, True), ("sl_actors", True, True),
    ("fin_budget_src", False, False), ("fin_revenue_src", False, False),
]
rows, counts = [], {}
for col, multi, hidden in cat_specs:
    row, vc = category_profile(pop[col], col, multi, names_hidden=hidden)
    row["role"] = role_of(col)
    rows.append(row)
    counts[col] = vc
prof_cat = pd.DataFrame(rows).set_index("field")
with pd.option_context("display.max_rows", 60, "display.max_colwidth", 40):
    display(prof_cat)

# Track-record sparsity (counts only): directors and companies with a single film in the population
for col, label in [("rt_director", "RT directors"), ("production_companies", "TMDB production companies")]:
    vc = counts[col]
    print(f"{label}: {len(vc):,} distinct; {(vc == 1).mean():.0%} appear in only one usable film")

# The Numbers 'Franchise' looks like several names run together (e.g. 'BatmanDark Knight Trilogy')
fr = pop["tn_Franchise"].dropna()
glued = fr.str.contains(r"[a-z][A-Z]").sum()
print(f"tn_Franchise: {len(fr):,} films; {glued:,} values look like names glued together (lowercase then uppercase)")
flag("The Numbers", "tn_Franchise", "parse error", f"{glued:,} of {len(fr):,} values look like several franchise names run together")

fig, axes = plt.subplots(2, 3, figsize=(15, 7), layout="constrained")
for ax, col in zip(axes.ravel(), ["genres", "tn_MPAA Rating", "tn_Source", "tn_Creative Type", "original_language", "fin_budget_src"]):
    count_plot(ax, counts[col], pop[col].notna().sum(), col, xlabel="% of films with the level")
fig.suptitle("Selected categories, usable films (dotted line = 5% rare-level flag)", x=0.01, ha="left", fontsize=11)
plt.show()

### 3e. Identifier and free-text fields
Uniqueness, duplicates and length only. After the build's one-row-per-film step, no key should repeat.

In [ ]:
id_cols = ["id", "imdb_id", "title", "original_title", "tn_Movie URL", "rt_id", "sl_rotten_tomatoes_link",
           "wd_item", "overview", "tagline", "sl_movie_info", "sl_critics_consensus", "merged_ids"]
prof_id = id_text_profile(pop, id_cols)
display(prof_id)
for key in ["id", "imdb_id", "tn_Movie URL", "rt_id", "sl_rotten_tomatoes_link", "wd_item"]:
    shared = prof_id.loc[key, "rows sharing a value"]
    if shared:
        flag("Unified", key, "duplicate key", f"{shared} usable films share a {key}")
same_ty = pop.duplicated(["title", "year"], keep=False)
print(f"usable films sharing title and year with another film: {same_ty.sum()}")
if same_ty.any():
    display(pop.loc[same_ty, ["id", "title", "year", "fin_budget", "fin_revenue", "runtime", "in_tmdb"]].sort_values("title"))

### 3f. Release dates
TMDB's date drives the year split. Release month is an allowed input. Jan 1 is checked as a placeholder.

In [ ]:
d = pop["release_dt"]
jan1 = ((d.dt.month == 1) & (d.dt.day == 1)).sum()
print(f"release_dt: {d.min().date()} to {d.max().date()}, missing {d.isna().sum()}, Jan 1 dates {jan1} ({jan1 / len(d):.1%})")
if jan1 / len(d) > 0.01:
    flag("Unified", "release_dt", "placeholder date?", f"{jan1} films dated Jan 1")

month_share = pop["release_month"].value_counts(normalize=True).sort_index() * 100
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.2), width_ratios=[2, 1], layout="constrained")
for p, g in pop.groupby("period", observed=True):
    yc = g["year"].value_counts().sort_index()
    a1.bar(yc.index, yc.values, color=SERIES[p], width=0.8, label=f"{p} ({len(g):,})")
a1.set_title("Usable films per release year"); a1.set_xlabel("release year"); a1.set_ylabel("films")
a1.legend(fontsize=7, frameon=False); a1.grid(axis="x", visible=False)
a2.bar(month_share.index, month_share.values, color=BLUE, width=0.7)
a2.axhline(100 / 12, color=INK2, ls=":", lw=1)
a2.set_xticks(range(1, 13), list("JFMAMJJASOND"))
a2.set_title("Release month (dotted line = even 8.3%)"); a2.set_ylabel("% of films"); a2.grid(axis="x", visible=False)
plt.show()

### 3g. Do the sources agree? (new)

Where two sources report the same thing, they should agree. Disagreement is either a wrong match or a data error, and it tells Phase 3 which source to trust. Agreement = within 15% for money, 5 minutes for runtime, 7 days for release dates, same label for MPAA.

In [ ]:
def agree_num(a, b, tol):
    m = pop[a].gt(0) & pop[b].gt(0)
    r = (pop.loc[m, a] / pop.loc[m, b] - 1).abs()
    return {"films with both": int(m.sum()), "% agree": round(100 * (r <= tol).mean(), 1),
            "median |diff| %": round(100 * r.median(), 1)}

def agree_abs(a, b, tol):
    x, y = pop[a], pop[b]
    m = x.gt(0) & y.gt(0) if pd.api.types.is_numeric_dtype(x) else x.notna() & y.notna()
    diff = (x[m] - y[m]).abs()
    if hasattr(diff, "dt"):
        diff = diff.dt.days
    return {"films with both": int(m.sum()), "% agree": round(100 * (diff <= tol).mean(), 1),
            "median |diff| %": None}

agreement = pd.DataFrame({
    "budget: TMDB vs The Numbers": agree_num("budget", "tn_Production Budget (USD)", 0.15),
    "budget: TMDB vs Wikidata": agree_num("budget", "wd_wd_budget_usd", 0.15),
    "budget: The Numbers vs Wikidata": agree_num("tn_Production Budget (USD)", "wd_wd_budget_usd", 0.15),
    "revenue: TMDB vs The Numbers worldwide": agree_num("revenue", "tn_Worldwide Gross (USD)", 0.15),
    "revenue: TMDB vs Wikidata box office": agree_num("revenue", "wd_wd_box_office_usd", 0.15),
    "domestic: The Numbers vs Box Office Mojo": agree_num("tn_Domestic Gross (USD)", "bom_domestic_gross", 0.15),
    "domestic: The Numbers vs RT box office": agree_num("tn_Domestic Gross (USD)", "rt_boxOffice_usd", 0.15),
    "runtime: TMDB vs The Numbers (min)": agree_abs("runtime", "tn_Running Time (minutes)", 5),
    "runtime: TMDB vs RT (min)": agree_abs("runtime", "rt_runtimeMinutes", 5),
    "release: TMDB vs The Numbers (days)": agree_abs("release_dt", "tn_Release Date", 7),
    "release: TMDB vs Box Office Mojo first day (days)": agree_abs("release_dt", "bom_first_date", 7),
}).T
display(agreement)
for k, r in agreement.iterrows():
    if r["% agree"] < 80:
        flag("Unified", k, "sources disagree", f"{r['% agree']}% agree across {r['films with both']:,} films")

mpaa = pop[["tn_MPAA Rating", "rt_rating", "sl_content_rating"]].dropna()
print(f"MPAA rating, The Numbers vs RT vs RT sl (films with all three: {len(mpaa):,}): "
      f"all three agree on {(mpaa.nunique(axis=1) == 1).mean():.0%}")

# Matt's label risk: TMDB "worldwide" revenue that equals RT domestic box office
dom = pop["rt_boxOffice_usd"].gt(0) & pop["revenue"].gt(0)
same_dom = dom & (pop["revenue"] / pop["rt_boxOffice_usd"]).between(0.98, 1.02)
fixable = same_dom & pop["tn_Worldwide Gross (USD)"].gt(pop["revenue"] * 1.02)
print(f"TMDB revenue within 2% of RT domestic: {same_dom.sum():,} films (Matt: 396). "
      f"The Numbers has a higher worldwide gross for {fixable.sum():,} of them.")
flag("Unified", "fin_revenue", "domestic-only revenue?",
     f"{same_dom.sum():,} films have TMDB revenue = RT domestic; The Numbers worldwide is higher for {fixable.sum():,}")

ratio = (pop["budget"] / pop["tn_Production Budget (USD)"]).where(pop["budget"].gt(0) & pop["tn_Production Budget (USD)"].gt(0))
fig, ax = plt.subplots(figsize=(6, 3.2), layout="constrained")
ax.hist(np.log10(ratio.dropna()).clip(-1.5, 1.5), bins=60, color=BLUE, edgecolor="white", linewidth=0.4)
ax.axvspan(np.log10(1 / 1.15), np.log10(1.15), color=BLUE_LIGHT, zorder=0)
ax.set_title(f"Budget, TMDB / The Numbers (n = {ratio.notna().sum():,}; shaded = within 15%)")
ax.set_xlabel("log10 ratio (0 = identical)"); ax.set_ylabel("films")
plt.show()

### 3h. Who has which data? (new)
Matt found the RT join covers big films better than small ones (58% of $1–5M vs 91% of $60M+). The same check for every source and key input, by period and budget tier. Uneven coverage means missing values aren't random, so Phase 3 needs "unknown" indicators, not dropped rows.

In [ ]:
pop["budget_tier"] = pd.cut(pop["fin_budget"], [1e6, 5e6, 20e6, 60e6, np.inf], right=False,
                           labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
cov_cols = {"TMDB genres": "genres", "The Numbers match": "tn_Movie Name", "Wikidata": "wd_item",
            "Box Office Mojo": "bom_title", "RT movies (director)": "rt_director", "RT writer": "rt_writer",
            "RT sl (actors)": "sl_actors", "MPAA (The Numbers)": "tn_MPAA Rating", "Source (The Numbers)": "tn_Source",
            "Franchise (The Numbers)": "tn_Franchise", "Critic reviews": "rtrev_n_reviews"}
cov = pd.concat({
    "period": pop.groupby("period", observed=True).agg(**{k: (v, lambda s: s.notna().mean()) for k, v in cov_cols.items()}),
    "budget tier": pop.groupby("budget_tier", observed=True).agg(**{k: (v, lambda s: s.notna().mean()) for k, v in cov_cols.items()}),
})
display((100 * cov).round(0).astype(int).astype(str) + "%")
gap = (cov.loc["budget tier"].max() - cov.loc["budget tier"].min()) * 100
for k, g in gap.items():
    if g > 20:
        flag("Unified", cov_cols[k], "coverage skews by budget", f"{k}: coverage gap of {g:.0f} points between budget tiers")

### 3i. Flag register
Every flag from 2d–3h plus the skew, rare-level and missingness flags from the profile tables. Hand-off to the readiness gate (step 12) and the cleaning ledger (step 13).

In [ ]:
reg = list(FLAGS)
for f_, r in prof_num.iterrows():
    if r["role"] in (A, CAND, O, "derived") and r["skew flag"]:
        reg.append({"file": "Unified", "field": f_, "issue": "skew > 0.5",
                    "evidence": f"skew {r['skew']:.2f}; median {r['median']:,.3g} vs mean {r['mean']:,.3g}"})
    if r["role"] in (A, CAND) and r["% missing"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "missing", "evidence": f"{r['% missing']}% missing"})
for f_, r in prof_cat.iterrows():
    if r["role"] not in (A, CAND):
        continue
    if r["levels < 5% of rows"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "rare levels",
                    "evidence": f"{r['levels < 5% of rows']:,} of {r['levels']:,} levels under 5% of rows"})
    if r["% missing"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "missing", "evidence": f"{r['% missing']}% missing"})
register = pd.DataFrame(reg).drop_duplicates(["file", "field", "issue"]).sort_values(["issue", "field"])
with pd.option_context("display.max_rows", 150, "display.max_colwidth", 100):
    display(register.reset_index(drop=True))
print(f"{len(register)} flags across {register['field'].nunique()} fields")
register.groupby("issue").size().rename("flags").to_frame()

### 3j. What the profile shows (facts from the outputs above, run 2026-11-23)

**Population and the label**
1. **6,838 usable films** (train 4,952 · test 1,264 · stress 622), up from Matt's 5,766. The test set grows 16% (1,086 → 1,264), which tightens the bootstrap CI on the evidence standard.
2. **The added films are mostly losers.** Films whose budget or revenue came from The Numbers or Wikidata are 83% losers at 2.5×, against 65% for Matt's TMDB-only films. Overall: 67.5% (Matt 64.6%). This is the S4 worry in reverse: the extra sources recover some of the flops TMDB was missing. The 20%-capacity ceiling drops slightly, from 31% to 30% of losers.
3. **S4 is still triggered.** 39.0% of films with a budget ≥ $1M still have no revenue from any source (Matt: 43.6%), and 52% in the stress years.
4. **Junk values grew with the population.** 269 films earn < 1% of budget (Matt 179), 35 report revenue < $1,000, 380 < $100,000. The lowest are the same junk Matt found (*Aquajots*: $100M budget, $1 revenue). Six films above 100×, including *Vacancy* (2018, $903M on $1M) and a new one, *Space Station 3D*, where a Wikidata budget meets a TMDB revenue.

**Do the sources agree?** (new)
5. **Money mostly agrees.** Budget: TMDB vs The Numbers 85%, TMDB vs Wikidata 90% (within 15%). Worldwide revenue: 81–87%. Domestic gross across The Numbers, Box Office Mojo and RT: 92–97%. Runtime: 92–93% within 5 minutes. MPAA rating: the three sources agree on 99%.
6. **Release dates often don't.** TMDB's date is within 7 days of The Numbers / Box Office Mojo for only 73–75% of films. TMDB often records the first release anywhere (a festival or a foreign market); the others record the US release. Release month is an allowed input, so Phase 3 has to pick one definition.
7. **Matt's domestic-as-worldwide risk is larger than first counted:** 434 films (Matt: 396) have TMDB revenue equal to RT domestic. **The Numbers has a higher worldwide figure for 220 of them**, a direct repair.

**Coverage is uneven** (new)
8. **Every extra source covers big films far better than small ones.** The Numbers matches 37% of $1–5M films vs 91% of $60M+; MPAA and source material 35% vs 91%; director 62% vs 90%; franchise 3% vs 44%. Small films are where most losers are, so a missing value is not random. RT stefanoleone992 (scraped in 2020) has actors for only 5% of stress-period films.
9. **Franchise is thin and partly garbled:** 15% coverage; 92 of 1,006 values look like several franchise names run together (e.g. "BatmanDark Knight Trilogy").
10. **Track records are sparse:** 63% of directors and 67% of production companies appear in only one usable film.

**Keys and duplicates**
11. **Duplicates (fixed in the build, 2026-11-23).** The first profile found title+year pairs that were the same film entered twice (*Spider-Man: No Way Home*, *Timber Falls*): the build's rule required runtime within 5 minutes, and one copy had runtime 0. The rule now treats 0 as unknown, which also caught *WALL·E* and *Black Panther: Wakanda Forever*. 3 pairs remain and are **not** duplicates: *Underdogs* (two different 2013 films), *Uncharted 2* (a video game, entered twice) and a junk *Blue Beetle* entry ($400M revenue, runtime 0). Those are cleaning issues. 2 usable films share a Wikidata item.
12. **Missed The Numbers matches (fixed in the build, 2026-11-23).** About 120 The Numbers films had been added as separate "not in TMDB" rows although TMDB had them under a differently written title (*Star Wars **Ep. VII**: The Force Awakens*, *Kill Bill: **Volume** 1*, *Harry Potter and the **Sorcerer's** Stone*). A second matching pass (build §4b) now joins them, which removed about 90 double-counted usable films (6,928 → 6,838). About 10 remain where the titles share almost nothing (*X-Men 2* vs *X2*, *F9: The Fast Saga* vs *F9*).
13. **Review copies (fixed in the build).** The stefanoleone992 review file had 120,945 rows identical in every column, and the andrezaza file 12,394 repeated review ids (each repeat identical in every column). Only those copies are dropped; every distinct review is kept. The individual reviews are now saved as their own table, `reviews_unified.csv` (one row per review, keyed by `film_uid`).

**Raw-file notes:** The Numbers has 150 release dates after the access date (up to 2068); none reach the usable population. RT sl reviews have 83 placeholder dates (1800-01-01). The Box Office Mojo scraper has no license.

### 3k. Proposed dispositions and phase gate (DRAFT, pending Tanner + Matt sign-off)

Matt's §3k rows 1–12 still apply. These are the additions for the unified dataset. **None are applied here.**

| # | Flag | Evidence | Proposed disposition | Treatment | Verification |
|---|---|---|---|---|---|
| U1 | Population now multi-source | 6,838 films; 1,078 use The Numbers / Wikidata financials | Requires a contract amendment | Dated amendment: population = 2000–2023, budget ≥ $1M, revenue > 0 **from TMDB, The Numbers or Wikidata**, with source precedence stated | Amendment recorded in the contract file |
| U2 | Added films skew to losers | 83% vs 65% losers | Usable; report it | Report results on the full population and on Matt's TMDB-only population as a sensitivity check | Capture on both populations |
| U3 | Domestic-as-worldwide revenue | 434 films; The Numbers higher for 220 | Requires repair | Replace TMDB revenue with The Numbers worldwide where TMDB = RT domestic and The Numbers is higher. Flag the other 214. | Loser rate before / after |
| U4 | Source disagreement on budget / revenue | 81–90% agree within 15% | Requires a rule | Keep precedence TMDB → The Numbers → Wikidata, but hand-check films where they disagree by > 2× | Hand-check sample error rate |
| U5 | Release date definitions differ | 74% agree within 7 days | Requires a rule | Use the US release date (The Numbers / Box Office Mojo) where available, TMDB otherwise; keep the source | Month distribution before / after |
| U6 | Coverage skews by budget | The Numbers 37% vs 91%; director 62% vs 90% | Usable with a limitation | Every optional input gets an "unknown" indicator; no dropped rows | Loser rate by known / unknown |
| U7 | Leftover duplicates | Runtime 0 blocked the merge rule | **Done** (build, 2026-11-23) | Runtime 0 counts as unknown; 4 more films merged | 3 same-title pairs left, none duplicates. Second pass (build §4b) joined ~120 missed The Numbers matches; ~10 remain (*X-Men 2* / *X2*) |
| U8 | Review copies | 120,945 + 12,394 copies of the same review | **Done** (build, 2026-11-23) | Only identical copies dropped; all reviews saved to `reviews_unified.csv` | Counts reported in the build |
| U11 | Junk entries | *Uncharted 2* (video game), *Blue Beetle* copy with $400M revenue and runtime 0 | Requires repair | Cleaning rule for runtime-0 TMDB entries that collide with a real film | Pair count after cleaning |
| U9 | Franchise field | 14% coverage, 92 garbled values | Investigate | See `FEATURE_IDEAS.md` (sequel / franchise); likely needs Wikidata series data | Coverage after |
| U10 | Candidate inputs not in the contract | MPAA, source, creative type, writer, cast, franchise | Needs a decision | Decide which to add by amendment before Phase 3 features | Decision recorded |

**Proposed phase gate: CONTINUE**, after U1 (amendment). Build fixes U7 and U8 are done. The S4 readiness verdict still belongs to step 12.

## 4. Missing values and repeated columns (step toward cleaning)

**Goal:** many facts appear in several columns, one per source (runtime is in four). Before combining them, measure what's actually missing in each, so a fake value (TMDB's `0`) never overrides a real one, and so we know whether a second source fills the gaps or misses the same films.

**Which rows:** the usable films. **This section only measures; nothing is changed.**

**What counts as missing** (found by scanning every column for placeholder values):

| Rule | Columns | Why |
|---|---|---|
| Blank (NaN) | all | — |
| `0` | TMDB `budget`, `revenue`, `runtime` | TMDB writes 0 for unknown (634 / 692 / 55 usable films) |
| `-` | `bom_distributor` | placeholder (57 films) |
| **Not missing** | "Not Rated" / "NR" ratings, 0% Tomatometer, 0 reviews before release | real values |

In [ ]:
ZERO_MISSING = {"budget", "revenue", "runtime"}
DASH_MISSING = {"bom_distributor"}

def present(c):
    """True where column c has a real value on the usable films (missing rules above)."""
    s = pop[c]
    ok = s.notna()
    if c in ZERO_MISSING:
        ok &= s.ne(0)
    if c in DASH_MISSING:
        ok &= s.astype(str).str.strip().ne("-")
    if s.dtype == object or str(s.dtype) == "str":
        ok &= s.astype(str).str.strip().ne("")
    return ok

# One fact, several source columns (first column listed = the source used today, if any)
CONCEPTS = {
    "budget":              ("allowed input",  ["budget", "tn_Production Budget (USD)", "wd_wd_budget_usd"]),
    "worldwide revenue":   ("label",          ["revenue", "tn_Worldwide Gross (USD)", "wd_wd_box_office_usd"]),
    "domestic gross":      ("post-release",   ["tn_Domestic Gross (USD)", "bom_domestic_gross", "rt_boxOffice_usd"]),
    "runtime":             ("allowed input",  ["runtime", "tn_Running Time (minutes)", "sl_runtime", "rt_runtimeMinutes"]),
    "release date":        ("allowed input",  ["release_date", "tn_Release Date", "bom_first_date", "sl_original_release_date", "rt_releaseDateTheaters"]),
    "genre":               ("allowed input",  ["genres", "tn_Genre", "sl_genres", "rt_genre"]),
    "MPAA rating":         ("candidate",      ["tn_MPAA Rating", "sl_content_rating", "rt_rating"]),
    "director":            ("allowed input",  ["rt_director", "sl_directors"]),
    "writer":              ("candidate",      ["rt_writer", "sl_authors"]),
    "cast":                ("candidate",      ["sl_actors"]),
    "production companies":("allowed input",  ["production_companies", "tn_Production/Financing Companies", "sl_production_company"]),
    "distributor":         ("candidate",      ["rt_distributor", "bom_distributor"]),
    "country":             ("allowed input",  ["production_countries", "tn_Production Countries"]),
    "language":            ("allowed input",  ["original_language", "rt_originalLanguage", "tn_Languages"]),
    "critic score":        ("post-release",   ["rt_tomatoMeter", "sl_tomatometer_rating", "rtrev_pct_fresh", "slrev_pct_fresh"]),
    "audience score":      ("post-release",   ["vote_average", "rt_audienceScore", "sl_audience_rating"]),
    "franchise":           ("candidate",      ["tn_Franchise"]),
    "source material":     ("candidate",      ["tn_Source"]),
    "creative type":       ("candidate",      ["tn_Creative Type"]),
}

rows = []
for concept, (role, cols) in CONCEPTS.items():
    P = pd.DataFrame({c: present(c) for c in cols})
    first = P.iloc[:, 0]
    rows.append({
        "concept": concept, "role": role, "columns": len(cols),
        "best single source": f"{P.mean().idxmax()} ({P.mean().max():.0%})",
        "first column": f"{first.mean():.0%}",
        "any source": f"{P.any(axis=1).mean():.0%}",
        "gain from combining": f"+{P.any(axis=1).mean() - P.mean().max():.0%}",
        "missing everywhere": int((~P.any(axis=1)).sum()),
    })
coverage = pd.DataFrame(rows).set_index("concept")
print(f"{len(pop):,} usable films; {sum(len(c) for _, c in CONCEPTS.values())} columns in {len(CONCEPTS)} concepts")
coverage

### 4a. Fill rate of every source column

Per concept: how full each source column is on the usable films, and how many films only that column covers (`only here`). A column that is never the only source adds nothing once the others are combined.

In [ ]:
detail = []
for concept, (role, cols) in CONCEPTS.items():
    P = pd.DataFrame({c: present(c) for c in cols})
    for c in cols:
        others = P.drop(columns=c).any(axis=1) if len(cols) > 1 else pd.Series(False, index=P.index)
        detail.append({"concept": concept, "column": c, "filled": f"{P[c].mean():.0%}",
                       "only here": int((P[c] & ~others).sum())})
pd.DataFrame(detail).set_index(["concept", "column"])

### 4b. Combine rules (reviewed by Tanner 2026-11-24; applied in `01_clean_dataset.ipynb`)

**General approach (agreed 2026-11-24):** each concept becomes **one combined column**. It takes the fullest source first, then fills its gaps from the others in the order listed. A value only counts if it's real under the §4 missing rules, so a TMDB `0` never wins. Each combined column gets a `_src` column recording which source supplied it. The original source columns move to a **source-detail** group: kept in the file for audits, but never model inputs. Applied in `01_clean_dataset.ipynb`, which writes `films_clean.csv`.

Fill rates are on the 6,838 usable films (§4 tables).

| # | Combined column | Fill order | Formatting before combining | Fill | Role | Notes |
|---|---|---|---|---|---|---|
| C1 | `fin_budget` | TMDB → The Numbers → Wikidata | none | 100% | allowed input | **Already built.** Keep. |
| C2 | `fin_revenue` | TMDB → The Numbers → Wikidata | none | 100% | label only | **Already built.** U3 (domestic-as-worldwide repair) is a separate cleaning step. |
| C3 | `domestic_gross` | The Numbers → Box Office Mojo → RT | none | 76% | post-release: checks only | Used for the S4 and U3 cross-checks, never as an input. |
| C4 | `runtime_min` | TMDB → RT → The Numbers → RT sl | none | 99% | allowed input | Values agree 92–93% within 5 min. 86 films left with none. |
| C5 | `release_date_us` | The Numbers → RT theatrical → Box Office Mojo first day | none | 78% | allowed input | **Decided (Tanner, 2026-11-24): use the US theatrical date**, because that's what a financier plans around. |
| C6 | `release_date_first` | TMDB | none | 98% | allowed input (fallback) | Release month = month of `release_date_us`, or `release_date_first` if that's missing; `_src` records which. |
| C7 | `genres` | TMDB → RT → The Numbers | map RT / The Numbers genre names onto TMDB's 19 genres (e.g. "Sci-fi" → "Science Fiction", "Thriller/Suspense" → "Thriller") | 100% | allowed input | TMDB already covers 97%; the others only fill 31 + 109 films. Multi-label, kept as a list. |
| C8 | `mpaa_rating` | The Numbers → RT → RT sl | standardize to G / PG / PG-13 / R / NC-17 / NR. The Numbers' run-on text ("NC-17for explicit…") becomes its leading rating; "Not Rated" and "NR" → NR; TV ratings and "Open" → missing | 76% | candidate input (A4) | Sources agree 99%. ⚠️ RT has **no NR level**, so a blank RT rating may really mean "not rated". "Unknown" and "NR" stay separate. |
| C9 | `director` | RT → RT sl | none (same name format) | 82% | allowed input (as **track record from earlier films only**) | 1,209 films have no director in any source. |
| C10 | `writer` | RT → RT sl | none | 81% | candidate input (A4) | Same track-record treatment as director. |
| C11 | ~~`cast`~~ | — | — | 60% | **dropped as an input** (Tanner, 2026-11-24) | Only 5% coverage in 2020–23 (2020 scrape). `sl_actors` stays in source detail. |
| C12 | `production_companies` | TMDB → The Numbers → RT sl | none: only fills TMDB's blanks, no name matching | 96% | allowed input (as track record) | Names differ by source ("Warner Bros." vs "Warner Bros. Pictures"). Standardizing names is a later cleaning step. |
| C13 | `distributor` | RT → Box Office Mojo | `-` → missing | 70% | candidate input (A4) | Same name-format issue as companies. |
| C14 | `countries` | TMDB → The Numbers | "United States" → "United States of America" (TMDB's names) | 98% | allowed input | Multi-label. |
| C15 | `original_language` | TMDB → RT | RT language name → ISO code ("English" → `en`) | 100% | allowed input | `tn_Languages` lists every spoken language, not the original one: source detail only. |
| C16 | `critic_pct_fresh` | RT reviews (`rtrev_pct_fresh`) → RT Tomatometer → RT sl Tomatometer | none (all 0–100) | 79% | **post-release: description only** | The one review-consensus column for the films table. `rtrev_n_reviews` stays next to it, as does `sl_critics_consensus` (text). |
| C17 | `audience_score` | TMDB `vote_average` × 10 → RT → RT sl | put on a 0–100 scale | 98% | **post-release: description only** | **Kept** (Tanner, 2026-11-24) to show how audience reception relates to money-losers. Never an input. |
| C18 | `franchise`, `source_material`, `creative_type` | The Numbers only | split the 92 run-together franchise names (U9) | 15% / 63% / 62% | candidate inputs (A4) | Nothing to combine. Franchise likely needs Wikidata series data (`FEATURE_IDEAS.md`). |

**Kept as is:** `film_uid`, `id`, `imdb_id`, `title`, `original_title`, `year`, `in_tmdb`, `usable`, `merged_ids`, all `fin_*`.

**Moves to source detail** (about 50 columns): every source's own copy of the facts above (`tn_Running Time (minutes)`, `sl_genres`, `bom_first_date`, …), match metadata (`*_year_diff`, `rt_rule`, `rt_matched_on`, source titles), and post-release columns not used for checks (DVD sales, opening weekend, theater counts, popularity, vote counts).

**"Unknown" indicators (U6):** every candidate or allowed input that is below 100% gets a missing flag (e.g. `mpaa_rating_missing`). Coverage depends on budget (§3h), so blanks aren't random and shouldn't be filled with a guess or dropped.

**Review decisions (Tanner, 2026-11-24):**
1. C5/C6: use the US theatrical date, with TMDB's date only as the fallback.
2. C11: cast is dropped as an input.
3. C17: audience score is kept, for correlation and description only.
All other rules are accepted as drafted.

## 5. The cleaned columns that could go into a model

From `films_clean.csv` (built by `01_clean_dataset.ipynb`), on the usable films: the budget and the combined input columns. Not shown: keys and ids (`film_uid`, `id`, `imdb_id`, titles), population flags, the label, review/description columns, `_src` / `_missing` helpers and the source-detail columns. They stay in the file for joins and audits but aren't model inputs.

In [ ]:
clean = pd.read_csv("../data/clean/films_clean.csv", low_memory=False)
cols = pd.read_csv("../data/clean/films_clean_columns.csv")
cu = clean[clean["usable"]]

def kind(c):
    s = cu[c].dropna()
    if len(s) and s.isin([True, False]).all():
        return "bool"
    if pd.api.types.is_numeric_dtype(cu[c]):
        return "number"
    if len(s) and pd.to_datetime(s.head(50), errors="coerce", format="mixed").notna().all():
        return "date"
    return "text"

cols["type"] = cols["column"].map(kind)
cols["filled"] = cols["column"].map(lambda c: int(cu[c].notna().sum()))
cols["blank"] = len(cu) - cols["filled"]
cols["fill %"] = (100 * cols["filled"] / len(cu)).round(0).astype(int)
cols["distinct"] = cols["column"].map(lambda c: cu[c].nunique())
core = cols[(cols["group"].eq("input") | cols["column"].eq("fin_budget"))
            & ~cols["column"].str.endswith(("_src", "_missing"))]
print(f"{len(core)} columns that could be model inputs (of {len(cols)} in the file)")
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 45):
    display(core.drop(columns="% filled (usable films)").set_index(["group", "column"]))

## 6. A closer look at six inputs

What the values are, and how they spread, for `runtime_min`, `genres_all`, `production_companies`, `distributor`, `countries` and `creative_type`. Usable films only, from `films_clean.csv`. **Observe only; nothing is changed.** Reviews come later.

For list columns (genres, companies, countries), a film can have several values, so shares add up to more than 100%.

In [ ]:
def items(col):
    """One row per (film, value) for a comma-separated list column."""
    s = cu[col].dropna().astype(str).str.split(",").explode().str.strip()
    return s[s.ne("")]


def per_film(col):
    """How many values each film has (0 = blank)."""
    return cu[col].fillna("").astype(str).map(lambda x: len([t for t in x.split(",") if t.strip()]))


def films_per_value(col):
    """For each distinct value, how many usable films it appears in, grouped into buckets."""
    vc = items(col).value_counts()
    b = pd.cut(vc, [0, 1, 4, 9, np.inf], labels=["1 film", "2-4 films", "5-9 films", "10+ films"])
    out = pd.DataFrame({"values": b.value_counts().sort_index(),
                        "films covered": vc.groupby(b, observed=False).sum()})
    out["% of values"] = (100 * out["values"] / len(vc)).round(1)
    return out


n_u = len(cu)
print(f"{n_u:,} usable films")

### 6a. `runtime_min` (minutes)

TMDB → RT → The Numbers → RT sl. Shown: summary statistics, the distribution, runtime bands, and the shortest and longest films (to spot errors).

In [ ]:
rt = cu["runtime_min"]
display(numeric_profile(cu, ["runtime_min"]))
profile_grid([(rt, "runtime_min", "minutes", False)], "films", ncols=1)
bands = pd.cut(rt, [0, 60, 80, 90, 100, 110, 120, 150, 180, np.inf],
               labels=["< 60", "60-79", "80-89", "90-99", "100-109", "110-119", "120-149", "150-179", "180+"], right=False)
display(bands.value_counts(dropna=False).sort_index().rename("films").to_frame()
        .assign(**{"% of films": lambda d: (100 * d["films"] / n_u).round(1)}))
show = ["title", "year", "runtime_min", "runtime_min_src", "fin_budget"]
print("Shortest:"); display(cu.nsmallest(8, "runtime_min")[show])
print("Longest:"); display(cu.nlargest(8, "runtime_min")[show])

### 6b. `genres_all`

TMDB's 19 genres, with RT / The Numbers filling TMDB's blanks.

In [ ]:
g = items("genres_all")
gc = g.value_counts()
display(pd.DataFrame({"films": gc, "% of films": (100 * gc / n_u).round(1)}))
print("Genres per film:"); display(per_film("genres_all").value_counts().sort_index().rename("films").to_frame())
print("Most common genre combinations:")
combo = cu["genres_all"].dropna().map(lambda x: ", ".join(sorted(t.strip() for t in x.split(","))))
display(combo.value_counts().head(15).rename("films").to_frame())
fig, ax = plt.subplots(figsize=(6, 5))
count_plot(ax, gc, n_u, "genres_all", top=19)
plt.show()

### 6c. `production_companies`

TMDB → The Numbers → RT sl. Companies only work as an input as a **track record** (how a company's earlier films did), so the key question is how many films each company has.

In [ ]:
pc = items("production_companies")
pcc = pc.value_counts()
print(f"{len(pcc):,} distinct companies across {cu['production_companies'].notna().sum():,} films")
print("Companies per film:"); display(per_film("production_companies").value_counts().sort_index().head(12).rename("films").to_frame())
print("How many films each company appears in:"); display(films_per_value("production_companies"))
print("Top 25 companies:"); display(pd.DataFrame({"films": pcc.head(25), "% of films": (100 * pcc.head(25) / n_u).round(1)}))
print("Same company, different spellings (example: names containing 'Warner' or 'Universal'):")
display(pcc[pcc.index.str.contains("Warner|Universal", case=False)].head(15).rename("films").to_frame())

### 6d. `distributor`

RT → Box Office Mojo. One value per film. Like companies, it would be used as a track record, or grouped into major studio vs independent.

In [ ]:
dc = cu["distributor"].value_counts()
print(f"{len(dc):,} distinct distributors across {cu['distributor'].notna().sum():,} films ({cu['distributor'].isna().mean():.0%} blank)")
print("How many films each distributor appears in:"); display(films_per_value("distributor"))
print("Top 25 distributors:"); display(pd.DataFrame({"films": dc.head(25), "% of films": (100 * dc.head(25) / n_u).round(1)}))
print("Same distributor, different spellings (examples):")
display(dc[dc.index.str.contains("Warner|Universal|Sony|Disney|Buena|Fox|Lionsgate|Lions Gate", case=False)].rename("films").to_frame())

### 6e. `countries`

TMDB → The Numbers. A film can have several production countries (co-productions).

In [ ]:
cc = items("countries").value_counts()
print(f"{len(cc):,} distinct countries")
print("Top 20:"); display(pd.DataFrame({"films": cc.head(20), "% of films": (100 * cc.head(20) / n_u).round(1)}))
n_c = per_film("countries")
has_us = cu["countries"].fillna("").str.contains("United States of America", regex=False)
kind_c = np.select([n_c.eq(0), has_us & n_c.eq(1), has_us, n_c.eq(1)],
                   ["blank", "US only", "US co-production", "one non-US country"], "non-US co-production")
display(pd.Series(kind_c).value_counts().rename("films").to_frame()
        .assign(**{"% of films": lambda d: (100 * d["films"] / n_u).round(1)}))
print("Countries per film:"); display(n_c.value_counts().sort_index().head(10).rename("films").to_frame())

### 6f. `creative_type`

The Numbers only (blank when the film isn't in The Numbers). Since The Numbers covers big films much better, the blank rate is shown by budget tier.

In [ ]:
ct = cu["creative_type"].value_counts(dropna=False)
display(pd.DataFrame({"films": ct, "% of films": (100 * ct / n_u).round(1)}))
tier = pd.cut(cu["fin_budget"], [0, 5e6, 20e6, 60e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
print("Blank by budget tier:")
display(cu["creative_type"].isna().groupby(tier, observed=True).mean().map("{:.0%}".format).rename("% blank").to_frame())
print("Creative type by budget tier (films with a value):")
display(pd.crosstab(cu["creative_type"], tier, normalize="columns").mul(100).round(1))

### 6g. What the six inputs show (run 2026-11-24)

1. **Runtime exposes non-films in the usable population.** 75 "films" run 180+ minutes, and the longest are TV series and video games with a runtime of 999 (*God of War*, *The Bear*, *Avatar: The Last Airbender*, *What If...?*, *Dragon Ball Z Kai*). Under 60 minutes are junk entries (*Anne Hathaway and Emily Blunt Hosting the Oscars*, 1 minute, "$30M"). These are cleaning items (extends U11): not feature films, so they don't belong in the population. Typical films: median 105 min, middle half 95–120.
2. **Genre is usable as is.** 19 genres, 99.5% filled, mostly 2–3 per film. Drama (48%) and Comedy (35%) dominate. *TV Movie* (14 films) is another sign of non-theatrical entries. *Documentary*, *Western* and *TV Movie* are under 2% each.
3. **Production companies: track records will be thin.** 8,393 companies; two-thirds appear in only one usable film, and only 382 appear in 10+. Name variants split the same studio ("Warner Bros. Pictures", "Warner Bros. Pictures Animation", "Warner Bros. Entertainment France"…). A track record needs either name standardization or a "major studio" grouping.
4. **Distributor is concentrated but messy.** 548 names, but 77 distributors with 10+ films cover 4,397 of the 4,811 films that have one. Same studio, several spellings ("Warner Bros." vs "Warner Bros. Pictures"; "Fox" vs "20th Century Fox"), and some values list two companies. 30% blank. A major / mini-major / independent grouping looks more useful than individual names.
5. **Countries: mostly US.** 65% of films include the US (42% US only, 23% US co-productions); 33% are non-US. A small set of flags (US / UK / France / India / co-production) would capture most of it; 108 countries are too many to use individually.
6. **Creative type is informative but its blanks aren't random.** 38% blank overall, but 64% for $1–5M films vs 9% for $60M+. Where filled, it tracks budget strongly (Super Hero and Kids Fiction are big-budget; Factual is small-budget). Using it needs the `creative_type_missing` flag. One stray value ("\tScience Fiction", a tab character) is fixed in `01_clean_dataset.ipynb`.

### 6h. Tanner's review of the six inputs (2026-11-24)

**Runtime**
- Use runtime to find and drop non-films: outliers (999 min, under 60 min) and TV shows.
- Identify TV shows with runtime **plus** other signals, e.g. the *TV Movie* genre. Then remove them in cleaning.

**Genres**
- Look at *Music*: take a sample to see whether these are music videos/concerts (drop) or feature films about musicians, like biopics (keep).
- Films in several genres are fine. How to represent combinations is still open.

**Production companies**
- Group name variants into one studio in cleaning (all Warner Bros. variants → Warner Bros.; same for Universal, etc.).
- Then categorize: studio groups, foreign studios, animation vs live-action arms.

**Distributor**
- Same as companies: standardize spelling in cleaning, then group.
- Some films list several distributors. Open: how to link multiple distributors to one film (like genres).

**Countries**
- Keep for now, but likely the first input to drop: most big-budget films are US.
- Better as a **US / non-US** category than 108 countries.

**Creative type**
- Interesting values, but 38% blank and overlaps genre (a finer-grained genre).
- Don't go find data to fill it. Usefulness doubtful; decide at modeling.

**Cleaning to-do from this review** (for `01_clean_dataset.ipynb`)
1. Drop runtime outliers and TV shows / non-films.
2. Sample *Music* genre; drop non-films.
3. Standardize production company and distributor names, then group.
4. Decide how to handle multiple distributors per film.


## 7. The remaining inputs

Same review as §6 for the other input columns: `fin_budget`, release timing, `mpaa_rating`, `director`, `writer`, `original_language`, `franchise`, `source_material`. Values and spread only; no outcome yet.

### 7a. `fin_budget`

The production budget (the contract allows it as an input). Shown: spread, budget tiers, round-number budgets, and the smallest and largest.

In [ ]:
b = cu["fin_budget"]
display(numeric_profile(cu, ["fin_budget"]))
profile_grid([(np.log10(b), "log10 fin_budget", "log10 $ (6 = $1M, 8 = $100M)", False)], "films", ncols=1)
btier = pd.cut(b, [0, 5e6, 20e6, 60e6, 150e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60-150M", "$150M+"])
display(btier.value_counts().sort_index().rename("films").to_frame().assign(**{"% of films": lambda d: (100 * d["films"] / n_u).round(1)}))
print(f"Budgets that are a round $1M: {(b % 1e6 == 0).mean():.0%}")
print("Source:"); display(cu["fin_budget_src"].value_counts().rename("films").to_frame())
show = ["title", "year", "fin_budget", "fin_budget_src", "runtime_min"]
print("Largest:"); display(cu.nlargest(8, "fin_budget")[show])

### 7b. Release timing: `release_date`, `release_month`, `release_quarter`

US release date, falling back to TMDB's first release. Shown: films by month and year, and how often the fallback is used.

In [ ]:
display(cu["release_date_src"].value_counts().rename("films").to_frame())
rm = cu["release_month"].value_counts().sort_index()
display(pd.DataFrame({"films": rm, "% of films": (100 * rm / n_u).round(1)}).T)
yr = cu["year"].value_counts().sort_index()
display(yr.rename("films").to_frame().T)
rd = pd.to_datetime(cu["release_date"], format="mixed", errors="coerce")
print(f"Release year differs from film year: {(rd.dt.year != cu['year']).mean():.1%} of films")
print("Day of week:"); display(rd.dt.day_name().value_counts().rename("films").to_frame().T)

### 7c. `mpaa_rating`

The Numbers → RT → RT sl, standardized. "NR" (not rated) is a real value; blank means no source had one.

In [ ]:
mr = cu["mpaa_rating"].value_counts(dropna=False)
display(pd.DataFrame({"films": mr, "% of films": (100 * mr / n_u).round(1)}))
print("Blank by budget tier:")
display(cu["mpaa_rating"].isna().groupby(btier, observed=True).mean().map("{:.0%}".format).rename("% blank").to_frame())
print("Blank by original language (top 6):")
top_lang = cu["original_language"].value_counts().head(6).index
display(cu[cu["original_language"].isin(top_lang)].groupby("original_language")["mpaa_rating"].apply(lambda s: f"{s.isna().mean():.0%}").rename("% blank").to_frame())

### 7d. `director` and `writer`

People only work as an input as a **track record** (how their earlier films did), so the key question is how many films each person has.

In [ ]:
for col in ["director", "writer"]:
    v = items(col).value_counts()
    print(f"--- {col}: {len(v):,} distinct people across {cu[col].notna().sum():,} films ---")
    print("People per film:"); display(per_film(col).value_counts().sort_index().head(8).rename("films").to_frame().T)
    print("How many films each person appears in:"); display(films_per_value(col))
    print("Top 10:"); display(v.head(10).rename("films").to_frame().T)
both = cu[["director", "writer"]].dropna()
same = both.apply(lambda r: bool(set(t.strip() for t in r["director"].split(",")) & set(t.strip() for t in r["writer"].split(","))), axis=1)
print(f"Director also credited as writer: {same.mean():.0%} of films with both")

### 7e. `original_language`

TMDB → RT, ISO codes.

In [ ]:
ol = cu["original_language"].value_counts(dropna=False)
display(pd.DataFrame({"films": ol, "% of films": (100 * ol / n_u).round(1)}).head(15))
print(f"{cu['original_language'].nunique()} distinct languages")
print("English vs countries (does the US flag already cover it?):")
us = cu["countries"].fillna("").str.contains("United States")
display(pd.crosstab(cu["original_language"].eq("en").map({True: "English", False: "Not English"}), us.map({True: "includes US", False: "no US"}), margins=True))

### 7f. `franchise`

The Numbers only, split into one value per franchise (`|`-separated). Blank means either "not in a franchise" or "not in The Numbers", and the two can't be told apart.

In [ ]:
fr = cu["franchise"].dropna().str.split("|").explode().str.strip()
frc = fr.value_counts()
print(f"{cu['franchise'].notna().mean():.1%} of films have a franchise; {len(frc):,} distinct franchises")
print("Films per franchise:")
display(pd.cut(frc, [0, 1, 2, 4, np.inf], labels=["1", "2", "3-4", "5+"]).value_counts().sort_index().rename("franchises").to_frame().T)
display(frc.head(15).rename("films").to_frame().T)
print("Filled by budget tier:")
display(cu["franchise"].notna().groupby(btier, observed=True).mean().map("{:.0%}".format).rename("% with franchise").to_frame().T)
print("Franchise vs source_material:")
display(pd.crosstab(cu["source_material"].fillna("(blank)"), cu["franchise"].notna().map({True: "franchise", False: "no franchise"})))

### 7g. `source_material`

The Numbers only.

In [ ]:
sm = cu["source_material"].value_counts(dropna=False)
display(pd.DataFrame({"films": sm, "% of films": (100 * sm / n_u).round(1)}))
print("Blank by budget tier:")
display(cu["source_material"].isna().groupby(btier, observed=True).mean().map("{:.0%}".format).rename("% blank").to_frame().T)
print(f"source_material and creative_type both blank: {(cu['source_material'].isna() & cu['creative_type'].isna()).mean():.0%}; "
      f"only one blank: {(cu['source_material'].isna() ^ cu['creative_type'].isna()).mean():.0%}")

### 7h. What the remaining inputs show (run 2026-11-24), for Tanner's review

1. **Budget has bad values at the top.** Most budgets look right: median $14M, 74% round millions. But the largest are errors: *Batman v Superman … Ultimate Edition* ($874M, a re-release copy), *Adventures in Bora Bora* ($800M, 5 min), *M4's The Hobbit Book Edit* ($700M, a fan edit), *Diggy and The Bandits* ($447M). *Lost in the Stars* ($418M, Chinese) is probably in yuan, not dollars, so **some non-US budgets may be in local currency**. Cleaning item: check large budgets and non-English films' budgets against a second source.
2. **Release timing is clean, but the timing rule needs a decision.** Releases are spread evenly, with October and December highest. 22% of dates are the TMDB fallback (not US), which explains the Wednesday/Thursday releases (most US releases are Friday). Films per year drop by half in 2020–23 (COVID). ⚠️ The release month is usually picked well after financing. Decision: allow it as "planned release window," or drop it.
3. **MPAA rating blanks follow language.** 24% blank overall: 6% for English films, 56–93% for non-English films. A blank mostly means "not released in the US with a rating." Keep `mpaa_rating_missing` as a flag, or merge blank with NR.
4. **Directors and writers: track records will be thin, like companies.** 63% of directors and 67% of writers appear in only one usable film; only about 30 people in each have 10+ films. 48% of films are directed by one of their writers. Better as "number of earlier films" (experience) than as names. Track records must only count films released **before** this one.
5. **Language mostly repeats country.** 72% English. English lines up with "includes US" for 89% of films. With a US / non-US flag, language adds little; an English / non-English flag at most. Note: TMDB has both `cn` (Cantonese) and `zh` (Mandarin).
6. **Franchise is useful but blank is ambiguous.** 15% of films have one; 530 franchises, MCU largest (33). Strongly tied to budget: 4% of $1–5M films vs 65% of $150M+. Blank can mean "not a franchise" or "not in The Numbers"; since nearly all usable films are in The Numbers, blank is close to "no franchise." Good base for the sequel idea in `FEATURE_IDEAS.md`. 478 franchise films are tagged "Original Screenplay," so the two columns measure different things.
7. **Source material repeats creative type's gaps.** 37% blank, 63% for $1–5M films. When one is blank, the other almost always is too (only 1% differ), so both come from the same missing The Numbers record. Too many small categories (23); group into Original / Book / Real life / Comic / Remake-sequel / Other.

**Cleaning to-do from this review** (adds to §6h)
5. Check and fix extreme budgets and possible local-currency budgets.
6. Drop re-release editions and fan edits (e.g. "Ultimate Edition", "Book Edit").

**Decisions for Tanner**
- Release month as an input: allow (planned window) or drop?
- MPAA blank: keep as its own value, or merge with NR?
- Language: keep alongside the US flag, or drop?


### 7i. Tanner's review of the remaining inputs (2026-11-24)

**Budget**
- Must be cleaned. Find bad budgets by comparing **budget vs runtime**: films far outside the normal band are the suspects.
- Compare budget by **country / region** to find budgets in local currency (e.g. *Lost in the Stars*).

**MPAA rating**
- Fill blanks with NR.
- But keep the blank flag: a blank/NR may signal an entry that isn't a theatrical film (leakage of non-films into the data). Check this during non-film cleanup.

**Director and writer**
- Not enough data to rank "valuable" directors, but we do have film counts.
- Every person listed has at least one film. Count each person's films, then aggregate across a film's directors and writers. (Cleaning step; counts must only use earlier films when used as a model input.)
- Possible smaller feature: the writer + director combination.

**Language and country**
- They serve the same purpose. Combine them: **country is primary**; when country is blank, fill it from language.

**Franchise**
- Turn into a category. Blanks are real ("not a franchise"), since most films aren't franchises. **Don't drop for incompleteness.**
- Franchise × source material could be a sub-predictor (e.g. franchise based on a comic vs original). The pool gets small, but worth trying as a sub-level.

**Release timing**
- Keep. The outcome is revenue, and the release month is known before revenue.
- The frozen contract already allows the planned release month. Tanner also proposes moving the timing rule to "anything controllable before release" (amendment A6).

**Cleaning to-do from this review** (adds to §6h and §7h)
7. Flag suspect budgets with budget vs runtime and budget by country/region; fix or drop.
8. Fill blank MPAA with NR; keep `mpaa_rating_missing`.
9. Build film counts per director and writer.
10. Fill blank country from language.
11. Franchise: blank = "no franchise".


## 8. Outliers and non-films (candidate flags)

Each rule below is a **candidate**. Nothing is dropped here. Each rule shows how many usable films it catches and a sample to check. Once Tanner approves a rule, it moves into `01_clean_dataset.ipynb` as an `exclude_reason` (the film stays in the file but is no longer `usable`).

In [ ]:
pd.set_option("display.max_colwidth", 60)
SHOW = ["title", "year", "runtime_min", "fin_budget", "fin_revenue", "genres_all", "countries", "original_language", "mpaa_rating"]
flags = pd.DataFrame(index=cu.index)

# Outside sources: a real theatrical film is almost always in at least one of these.
cu = cu.assign(n_sources=cu["tn_Movie URL"].notna().astype(int) + cu["rt_id"].notna() + cu["bom_title"].notna() + cu["sl_rotten_tomatoes_link"].notna())

def check(rule, mask, note, n=15, sort=None, seed=0):
    flags[rule] = mask.reindex(cu.index, fill_value=False)
    hit = cu[flags[rule]]
    print(f"{rule}: {len(hit):,} usable films ({len(hit) / n_u:.1%}). {note}")
    hit = hit.sort_values(sort, ascending=False) if sort else hit.sample(min(n, len(hit)), random_state=seed)
    display(hit[SHOW + ["n_sources"]].head(n))

### 8a. Non-film signals

In [ ]:
check("runtime_400plus", cu["runtime_min"] >= 400, "TV series and video games (runtime 999 = 'unknown/very long').", sort="runtime_min")

In [ ]:
check("runtime_under_60", cu["runtime_min"] < 60, "Shorts, clips and junk entries.", n=25)

In [ ]:
check("tv_movie_genre", cu["genres_all"].fillna("").str.contains("TV Movie"), "TMDB's TV Movie genre.", n=20)

In [ ]:
pat = (r"\b(?:edition|extended|ultimate cut|director'?s cut|re-?release|original video|ufc \d|wwe|live (?:at|with|from)|live in \w+|"
       r"anniversary tour|confessions tour|in concert|concert|broadway|the play|soundstage|making of|behind the scenes|season \d)\b")
check("title_pattern", cu["title"].str.contains(pat, case=False, regex=True),
      "Title says it's an edition, event, concert, stage play or episode.", n=30)

In [ ]:
music = cu[cu["genres_all"].fillna("").str.contains("Music")]
print(f"Music genre: {len(music):,} usable films; Music as the only genre: {(music['genres_all'].str.strip() == 'Music').sum()}")
print("Random sample of 30 (Tanner's question: music videos/concerts, or films about musicians?):")
display(music.sample(30, random_state=1)[SHOW + ["n_sources"]])
check("music_only_genre", cu["genres_all"].fillna("").str.strip().eq("Music"), "Music is the only genre (likely concerts).", n=20)

In [ ]:
print("Usable films by number of outside sources (The Numbers, RT, RT sl, Box Office Mojo):")
display(cu["n_sources"].value_counts().sort_index().rename("films").to_frame().T)
check("no_outside_source", cu["n_sources"].eq(0), "Only TMDB knows this film; budget and revenue are unverified.", n=30, seed=2)
print("Original language of films with no outside source:")
display(cu.loc[flags["no_outside_source"], "original_language"].value_counts().head(8).rename("films").to_frame().T)

### 8b. Budget and revenue errors

Tanner's idea: compare **budget vs runtime** (a long, cheap film or a short, expensive one stands out) and **budget by country** (non-US budgets that look too large may be in local currency).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ok = cu["n_sources"] > 0
ax.scatter(cu.loc[ok, "runtime_min"], cu.loc[ok, "fin_budget"], s=4, alpha=.3, label="in an outside source")
ax.scatter(cu.loc[~ok, "runtime_min"], cu.loc[~ok, "fin_budget"], s=6, alpha=.6, color="C3", label="TMDB only")
ax.set(xscale="log", yscale="log", xlabel="runtime (min, log)", ylabel="budget ($, log)", title="Budget vs runtime, usable films")
ax.legend(); plt.show()
cu = cu.assign(budget_per_min=cu["fin_budget"] / cu["runtime_min"])
print("Budget per minute, by runtime band (typical films):")
bpm = cu.groupby(bands.reindex(cu.index), observed=True)["budget_per_min"].describe(percentiles=[.5, .95])[["count", "50%", "95%", "max"]]
display(bpm.astype({"count": int}).round({"50%": -3, "95%": -3, "max": -3}))
check("budget_per_min_extreme", (cu["budget_per_min"] > 3e6) | ((cu["runtime_min"] < 70) & (cu["fin_budget"] >= 20e6)),
      "Over $3M per minute, or under 70 min with a $20M+ budget.", n=25, sort="budget_per_min")

In [ ]:
country1 = cu["countries"].fillna("(blank)").str.split(",").str[0].str.strip()
by = cu.groupby(country1)["fin_budget"].agg(films="size", median="median", p90=lambda s: s.quantile(.9), max="max")
print("Budget by first-listed country (countries with 40+ films):")
display(by[by["films"] >= 40].sort_values("median", ascending=False).round({"median": -5, "p90": -5, "max": -5}))
non_us = ~cu["countries"].fillna("").str.contains("United States") & cu["original_language"].ne("en")
check("non_us_budget_100m", non_us & (cu["fin_budget"] >= 100e6), "Non-US, non-English films with a $100M+ budget: possibly local currency.", n=30, sort="fin_budget")

In [ ]:
check("revenue_equals_budget", cu["fin_revenue"].eq(cu["fin_budget"]), "Revenue exactly equals budget: a copied value, not a real figure.", n=20)
check("revenue_under_10k", cu["fin_revenue"] < 1e4, "Revenue under $10K (e.g. 5 or 10 dollars).", n=20, sort="fin_budget")

### 8c. Summary of candidate flags

In [ ]:
summary = pd.DataFrame({"films": flags.sum(), "% of usable": (100 * flags.sum() / n_u).round(1),
                        "also TMDB-only": flags.apply(lambda c: (c & flags["no_outside_source"]).sum())})
display(summary)
hard = flags.drop(columns="no_outside_source").any(axis=1)
print(f"Caught by at least one rule other than 'no outside source': {hard.sum():,} ({hard.mean():.1%})")
print(f"Plus 'no outside source': {flags.any(axis=1).sum():,} ({flags.any(axis=1).mean():.1%})")

### 8d. What the outlier checks show (run 2026-11-24), for Tanner's review

**The biggest finding: 929 usable films (14%) exist only in TMDB.** None of The Numbers, Rotten Tomatoes or Box Office Mojo has them, so their budget and revenue (and so the label) come only from TMDB's user-edited data. Almost every other rule mostly catches these same films. They are a mix:
- Junk: *Skibidi Toilet: The Movie*, *Adventures in Bora Bora* ($800M, 5 min), *😤 insect james 😤*, fan edits, concert videos.
- Real films, mostly non-English (80%): *Deep Sea* (China), *Death Note: The Last Name* (Japan), *Solomon Kane*.

| Rule | Films | Proposed |
|---|---|---|
| Runtime 400+ min | 12 | **Exclude.** All TV series or video games (*God of War*, *The Bear*, *What If...?*). |
| Runtime under 60 min | 46 | **Exclude.** Anime episodes, clips, junk. A few real IMAX documentaries (*Sea Rex*), which aren't feature films either. |
| TV Movie genre | 14 | **Exclude.** Not theatrical (*High School Musical*, *Stargate: Continuum*); their revenue figures are made up ($5,940). |
| Title says edition / cut / concert / play | 26 | **Exclude**, except theatrical concert films (*Hannah Montana … Concert*, *Glee: The Concert Movie*). Re-release editions duplicate the real film. |
| Music as the only genre | 15 | **Exclude** if TMDB-only (concert videos). |
| No outside source | 929 | **Decision needed** (below). |
| Budget per minute extreme | 25 | **Exclude.** Junk entries; also shows wrong runtimes on real titles. |
| Non-US $100M+ budget | 12 | **Check by hand.** Some are real US dollars (*The Battle at Lake Changjin*, *Detective Chinatown 3*); *Lost in the Stars* and *Chasing the Dragon* look like local currency. |
| Revenue = budget | 18 | **Exclude.** Copied value, not a real revenue. |
| Revenue under $10K | 101 | Keep if an outside source has the film (real tiny releases, e.g. *The Fanatic*); exclude if TMDB-only. |

**Tanner's Music question:** the Music genre is mostly real films about music (*A Star Is Born*, *Crazy Heart*, *Step Up*, *Footloose*). Concert films that played in theaters (*This Is It*, *Justin Bieber: Believe*) are real releases with real box office. Only the TMDB-only concert videos are junk. **Keep the genre; drop the videos.**

**Budget vs runtime:** works for junk at the edges (5-minute films with $100M budgets) but not in the middle. Budget per minute is roughly flat from 80 to 180 minutes.

**Budget by country:** doesn't show currency errors as a group. China has the highest median ($25M). Currency mistakes are individual films, so they need a hand check, not a country rule.

**New matching problem found:** in the TMDB snapshot, the real *Wonka* and *Wish* (2023) are still "Post Production", so the build skipped them. The Numbers' figures for them then matched to junk TMDB entries with the same title and year: *WONKA* (a 11-minute "Documentary") and *Wish* (a 66-minute drama). The rows have the real budget and revenue but the wrong runtime, genre and metadata. Fix in `00_build_dataset.ipynb`: don't match to TMDB entries whose runtime is far from the other source's, and consider late-2023 "Post Production" entries. Other late-2023 releases may have the same problem.

**Decision for Tanner: the 929 TMDB-only films**
1. **Exclude all** (recommended). The label can't be checked against anything. Cost: loses some real foreign films and makes the data more US-heavy.
2. **Exclude only those caught by another rule** (about 120). Keeps real foreign films, but leaves unverified revenue in the label.
3. **Keep all**, and add `n_sources` as a data-quality flag for a sensitivity check.

**After this:** §9 goes through missing values one column at a time.


### 8e. Tanner's decisions on the outlier rules (2026-11-24)

| Rule | Decision | Left after dropping TMDB-only |
|---|---|---|
| No outside source (929) | **Drop all.** "They do not feel like real movies." | 0 |
| *The Bear* | **Drop everywhere** (`tmdb-1233925`, `tmdb-1769662`). A TV series with odd numbers. | 0 |
| Runtime 400+ | **Drop** TV series and video games (confirmed 2026-11-24). Tanner first thought *Silent Hill 2* and *10 Things I Hate About You* were films; these entries are the 2001 video game and the 2009 TV series (the films are 2006 / 1999). All are TMDB-only, so dropped anyway. | 1 (*God of War*) |
| TV Movie genre | **Drop** (revised 2026-11-24). A TV movie has no box office, so its revenue and loser label can't be trusted. | 0 |
| Concerts / music events | **Keep if they had a theatrical run.** If people paid to see it, it's a movie (e.g. *Taylor Swift: The Eras Tour*). Having an outside source counts as a theatrical run. | 2 (*Hannah Montana*, *Glee*) |
| Extended editions / director's cuts | **Drop.** They come out after the original, usually without a theatrical run. | 1 (*Napoléon* director's cut) |
| Music genre | **Keep.** Mostly real films. | all |
| Budget per minute extreme | **Drop**, except *Winnie the Pooh* (2011) and *Pooh's Heffalump Movie* (real, double-check). | 7 |
| Revenue = budget | **Drop.** Data-entry errors; can't trust the numbers. Only *High School Musical 2* gave pause. | 9 |
| Runtime under 60 | **Drop all** (2026-11-24). | 0 |
| Non-US $100M+ budget | Hand check, not reviewed yet. | 6 |
| Revenue under $10K | Not reviewed yet (proposed keep). | 75 |

**TV movies:** Tanner first said keep, then agreed to drop after seeing that their "revenue" isn't box office (*High School Musical*: $5,940; *HSM 2*: revenue = budget).

**Build fix needed (`00_build_dataset.ipynb`):** the TMDB snapshot marks 46 films with 2023 release dates as "Post Production" or "In Production", so the build skipped them. Among them: *The Marvels* ($275M), *Wish*, *Wonka*, *Rebel Moon*, and *Taylor Swift: The Eras Tour*. Their The Numbers figures then matched junk TMDB entries with the same title (*WONKA*, *Wish*, likely *Five Nights at Freddy's*, 7 min). Fix: include those films, and don't match when runtimes disagree badly. The Numbers already has 5 of the 46 (*The Marvels*, *Wish*, *Wonka*…). The rest, including *Eras Tour*, are in no source. Tanner wants *The Marvels* (flop) in the data. Decision: recover the 5 The Numbers covers; no hand-entered data, so *Eras Tour* stays out.


### 8f. Budget fixes, currency check, and tiny revenues (agreed 2026-11-24)

Works on the films that survive the drops agreed in §8e (`keep`). Tanner agreed:
1. When TMDB's budget and The Numbers' (or Wikidata's) differ by more than 2×, use The Numbers.
2. For non-US films with only one budget source: chart budget vs revenue, and for the extreme ones, test "what if this budget is in the local currency?" using the country of origin. Worst case, drop them.
3. Revenue under $10K: keep films with a real MPAA rating; drop NR and blank.

In [ ]:
keep = (cu["n_sources"] > 0) & (cu["runtime_min"].fillna(999) >= 60) & (cu["runtime_min"].fillna(0) < 400) \
       & ~flags["tv_movie_genre"] & ~flags["revenue_equals_budget"] \
       & ~(flags["title_pattern"] & ~cu["title"].str.contains("concert", case=False)) \
       & ~(flags["budget_per_min_extreme"] & ~cu["title"].isin(["Winnie the Pooh", "Pooh's Heffalump Movie"]))
k = cu[keep].copy()
print(f"Usable films after the §8e drops: {len(k):,} (dropped {(~keep).sum():,})")

# 1. Two budget sources that disagree
tmdb_b = k["budget"].where(k["budget"] > 0)
other_b = k["tn_Production Budget (USD)"].fillna(k["wd_wd_budget_usd"])
ratio = tmdb_b / other_b
k["budget_fixed"] = np.where((ratio > 2) | (ratio < 0.5), other_b, k["fin_budget"])
fixed = k["budget_fixed"].ne(k["fin_budget"])
print(f"\n1. Films with two budgets: {ratio.notna().sum():,}; differ by more than 2x: {fixed.sum()} -> use The Numbers / Wikidata")
display(k.loc[fixed, ["title", "year", "countries", "fin_budget", "budget_fixed"]].assign(ratio=ratio[fixed].round(1))
        .sort_values("ratio", ascending=False).head(15))

In [ ]:
# 2. Non-US films with only one budget source
non_us = ~k["countries"].fillna("").str.contains("United States")
single = non_us & (tmdb_b.isna() | other_b.isna())
country1 = k["countries"].fillna("(blank)").str.split(",").str[0].str.strip()
print(f"Non-US films: {non_us.sum():,}; with only one budget source: {single.sum():,}")

fig, ax = plt.subplots(figsize=(8, 5))
for m, lab, col in [(~non_us, "US", "C0"), (non_us & ~single, "non-US, two budget sources", "C2"), (single, "non-US, one budget source", "C3")]:
    ax.scatter(k.loc[m, "budget_fixed"], k.loc[m, "fin_revenue"], s=5, alpha=.35, color=col, label=lab)
ax.plot([1e6, 1e9], [2.5e6, 2.5e9], "k--", lw=1, label="revenue = 2.5x budget")
ax.set(xscale="log", yscale="log", xlabel="budget ($, log)", ylabel="revenue ($, log)", title="Budget vs revenue")
ax.legend(fontsize=8); plt.show()

# Extreme = above twice the 95th percentile of verified budgets from the same country (or $100M+ if the country has none)
verified = k[non_us & ~single]
p95 = verified.groupby(country1[verified.index])["budget_fixed"].quantile(.95)
limit = country1.map(p95).fillna(50e6) * 2
extreme = single & (k["budget_fixed"] > limit) & (k["budget_fixed"] >= 20e6)

# Approximate average exchange rates, local currency per US$ (2000-2023, rounded). Only used for this test.
RATE = {"China": ("CNY", 6.8), "Hong Kong": ("HKD", 7.8), "Japan": ("JPY", 110), "South Korea": ("KRW", 1150),
        "India": ("INR", 65), "Russia": ("RUB", 50), "United Kingdom": ("GBP", 0.7), "Canada": ("CAD", 1.25),
        "Australia": ("AUD", 1.35), "France": ("EUR", 0.88), "Germany": ("EUR", 0.88), "Italy": ("EUR", 0.88),
        "Spain": ("EUR", 0.88), "Belgium": ("EUR", 0.88), "Mexico": ("MXN", 17), "Brazil": ("BRL", 3.5),
        "Turkey": ("TRY", 5), "Norway": ("NOK", 8), "Sweden": ("SEK", 8.5), "Denmark": ("DKK", 6.5),
        "Thailand": ("THB", 33), "Taiwan": ("TWD", 31), "Argentina": ("ARS", 20), "Poland": ("PLN", 3.6)}
x = k[extreme].assign(country=country1[extreme])
x["currency"] = x["country"].map(lambda c: RATE.get(c, ("?", np.nan))[0])
x["if_local_usd"] = x["budget_fixed"] / x["country"].map(lambda c: RATE.get(c, ("?", np.nan))[1])
x["country_p95"] = x["country"].map(p95)
x["local_plausible"] = x["if_local_usd"].between(1e6, x["country_p95"].fillna(50e6) * 2)
print(f"Extreme one-source budgets: {len(x)}")
display(x[["title", "year", "country", "budget_fixed", "fin_revenue", "country_p95", "currency", "if_local_usd", "local_plausible"]]
        .sort_values("budget_fixed", ascending=False))

In [ ]:
# 3. Revenue under $10K: keep only films with a real MPAA rating
tiny = k["fin_revenue"] < 1e4
rated = k["mpaa_rating"].isin(["G", "PG", "PG-13", "R", "NC-17"])
print(f"Revenue under $10K: {tiny.sum()} films; keep (rated) {(tiny & rated).sum()}, drop (NR or blank) {(tiny & ~rated).sum()}")

# New finding: "worldwide" revenue that is really the US gross
dom = k["domestic_gross"]
us_only = dom.notna() & (k["fin_revenue"] - dom).abs().le(0.01 * k["fin_revenue"])
loser = k["fin_revenue"] < 2.5 * k["budget_fixed"]
print("\nWorldwide revenue exactly equal to the US gross:")
display(pd.DataFrame({"films": [us_only[~non_us].sum(), us_only[non_us].sum()],
                      "% of group": [f"{us_only[~non_us].mean():.0%}", f"{us_only[non_us].mean():.0%}"],
                      "loser rate, these films": [f"{loser[us_only & ~non_us].mean():.0%}", f"{loser[us_only & non_us].mean():.0%}"],
                      "loser rate, everyone else": [f"{loser[~us_only & ~non_us].mean():.0%}", f"{loser[~us_only & non_us].mean():.0%}"]},
                     index=["US films", "non-US films"]))
display(k[us_only & non_us].sort_values("budget_fixed", ascending=False)[["title", "year", "countries", "budget_fixed", "fin_revenue", "fin_revenue_src"]].head(12))

### 8g. What 8f shows (run 2026-11-24)

1. **Budget disagreements:** 103 of 3,915 films with two budgets differ by more than 2×, and now use The Numbers. Most are extra-zero typos in TMDB (*Awake* $86M → $8.6M), in US and foreign films alike.
2. **The currency test doesn't work.** Only 11 one-source foreign budgets are extreme. Converting them "as local currency" gives impossible results for India (*RRR* would cost $1M; it cost about $70M). Most of the 11 are real dollar budgets of big hits (*RRR*, *Pathaan*, *2.0*, *Saaho*, *The Battle at Lake Changjin*). Currency errors look rare; typos are the real problem, and rule 1 handles those where a second source exists.
3. **Revenue under $10K:** 71 films left. Keep the 49 with a real MPAA rating; drop the 22 NR or blank.
4. **New problem: "worldwide" revenue that is only the US gross.** 618 films (515 US, 103 non-US) have revenue exactly equal to their US box office, so foreign box office is missing. They're labeled losers far more often (89% vs 62% for US films; 98% vs 70% non-US). For foreign films this is clearly wrong: *Nomad* shows $79K, *Kung Fu Jungle* $129K. For small US films it may be true (US-only releases), but probably not always.

**Decisions for Tanner**
- The 11 extreme foreign budgets: keep the known hits, drop the rest, or drop all 11?
- Revenue = US gross: drop the 103 foreign films? For the 515 US films, keep them but flag them for a sensitivity check?


### 8h. Tanner's decisions on 8g (2026-11-24)

- **Units:** all money is already in US dollars. Display in **$ thousands**.
- **Foreign currency:** none of the 11 extreme budgets is confirmed foreign currency. Drop the 4 that look wrong: *Chasing the Dragon*, *A Aa*, *Pink*, *Burnout*. Keep the real big-budget hits (*RRR*, *Pathaan*, *2.0*, *Saaho*, *The Battle at Lake Changjin*…).
- **Label:** change to **US box office < 1.25× budget** (about 67% losers). Population becomes films released in the US. This also fixes the 618 "worldwide = US gross" films. Drafted as amendment **A7** in `A10_contract_amendments_PROPOSED.md` (needs Matt).


## 9. Missing values, column by column

Population: films left after the §8 drops, released in the US (have a US box office), minus the 4 wrong foreign budgets (§8h). Money shown in **$ thousands**.

For each input: how many films are missing it, the share of the population, and (except for columns that already have a fill rule) a sample of the missing films. Tanner's hypothesis: many of these can be filled by looking the film up.

In [ ]:
pop = k[k["domestic_gross"].notna() & ~k["title"].isin(["Chasing the Dragon", "A Aa", "Pink", "Burnout"])].copy()
pop["budget_$k"] = (pop["budget_fixed"] / 1e3).round(0)
pop["us_gross_$k"] = (pop["domestic_gross"] / 1e3).round(0)
n_pop = len(pop)
print(f"Population: {n_pop:,} films")

INPUTS = ["runtime_min", "release_date", "genres_all", "mpaa_rating", "director", "writer", "production_companies",
          "distributor", "countries", "original_language", "franchise", "source_material", "creative_type"]
RULE = {"franchise": "blank = no franchise", "mpaa_rating": "blank -> NR, keep flag", "countries": "fill from original_language"}
miss = pd.DataFrame({"missing": pop[INPUTS].isna().sum()})
miss["% of population"] = (100 * miss["missing"] / n_pop).round(1)
miss["fill rule"] = miss.index.map(lambda c: RULE.get(c, ""))
display(miss.sort_values("missing", ascending=False))

BASE = ["title", "year", "budget_$k", "us_gross_$k", "countries"]
def show_missing(col, also=(), n=20, seed=0):
    m = pop[col].isna()
    print(f"{col}: {m.sum():,} missing ({m.mean():.1%} of {n_pop:,})")
    if col in RULE:
        print(f"  Fill rule: {RULE[col]}. No sample.")
        return
    cols = list(dict.fromkeys(BASE + list(also)))
    display(pop.loc[m, cols].sample(min(n, m.sum()), random_state=seed).sort_values("budget_$k", ascending=False))

### 9a. `creative_type`

In [ ]:
show_missing("creative_type", also=['genres_all', 'source_material'])

### 9b. `source_material`

In [ ]:
show_missing("source_material", also=['genres_all', 'creative_type'])

### 9c. `distributor`

In [ ]:
show_missing("distributor", also=['production_companies'])

### 9d. `writer`

In [ ]:
show_missing("writer", also=['director'])

### 9e. `director`

In [ ]:
show_missing("director", also=['writer'])

### 9f. `production_companies`

In [ ]:
show_missing("production_companies", also=['distributor'])

### 9g. `original_language`

In [ ]:
show_missing("original_language", also=[])

### 9h. `runtime_min`

In [ ]:
show_missing("runtime_min", also=['genres_all'])

### 9i. `mpaa_rating`

In [ ]:
show_missing("mpaa_rating", also=[])

### 9j. `countries`

In [ ]:
show_missing("countries", also=[])

### 9k. `franchise`

In [ ]:
show_missing("franchise", also=[])

### 9l. Filling from Wikidata (agreed 2026-11-24)

Snapshot `data/raw/wikidata/wikidata_film_people_2026-11-24.csv` (from `tools/get_wikidata_people.py`, matched by TMDB id). A feasibility test on 200 films showed director and writer agree with ours 96–97% of the time and runtime within 5 min 96%. Distributor was rejected (Wikidata lists worldwide and streaming distributors). Plan:
1. Fill blank `director`, `writer`, `runtime_min`.
2. Fill blank `source_material` only where Wikidata says what the film is based on.
3. Flag films where both have a director and they don't share one: a possible wrong match in the build.

In [ ]:
wd = pd.read_csv("../data/raw/wikidata/wikidata_film_people_2026-11-24.csv").set_index("tmdb_id")
w = wd.reindex(pop["id"].astype("Int64")).set_index(pop.index)
print(f"Population films found in Wikidata: {w['item'].notna().sum():,} of {n_pop:,}")

def names(s):
    return {t.strip().lower() for t in str(s).replace("|", ",").split(",") if t.strip() and str(s) != "nan"}

before = pop[["director", "writer", "runtime_min", "source_material"]].isna().sum()
pop["director"] = pop["director"].fillna(w["wd_director"].str.replace("|", ", "))
pop["writer"] = pop["writer"].fillna(w["wd_writer"].str.replace("|", ", "))
pop["runtime_min"] = pop["runtime_min"].fillna(w["wd_runtime_min"])

def source_from_type(t):
    t = str(t).lower()
    for keys, val in [(("comic", "manga", "graphic novel"), "Based on Comic/Graphic Novel"),
                      (("video game",), "Based on Game"),
                      (("television series", "tv series", "television program"), "Based on TV"),
                      (("musical",), "Based on Musical or Opera"), (("play",), "Based on Play"),
                      (("fairy tale", "folk", "legend", "myth"), "Based on Folk Tale/Legend/Fairytale"),
                      (("film",), "Remake"),
                      (("novel", "short story", "novella", "book", "literary work", "written work"), "Based on Fiction Book/Short Story")]:
        if any(k in t for k in keys):
            return val
    return np.nan
pop["source_material"] = pop["source_material"].fillna(w["wd_based_on_type"].map(source_from_type))
after = pop[["director", "writer", "runtime_min", "source_material"]].isna().sum()
display(pd.DataFrame({"missing before": before, "filled from Wikidata": before - after, "missing after": after,
                      "% missing after": (100 * after / n_pop).round(1)}))

In [ ]:
# Possible wrong matches: both sources name a director, and none overlap
ours = cu.loc[pop.index, "director"]          # our original value, before filling
both = ours.notna() & w["wd_director"].notna()
import unicodedata, difflib
def norm(n):
    n = unicodedata.normalize("NFKD", n).encode("ascii", "ignore").decode()
    return "".join(ch for ch in n.lower() if ch.isalpha() or ch == " ").split()
def same_person(a, b):
    # ignoring spaces, punctuation, accents and name order: same name, one name inside the other
    # ("Tarsem" / "Tarsem Singh"), same last name, or a near-identical spelling
    A, B = norm(a), norm(b)
    if not A or not B:
        return False
    if "".join(A) == "".join(B) or set(A) <= set(B) or set(B) <= set(A) or A[-1] == B[-1]:
        return True
    return difflib.SequenceMatcher(None, " ".join(sorted(A)), " ".join(sorted(B))).ratio() >= 0.8
def overlap(a, b):
    return any(same_person(x, y) for x in names(a) for y in names(b))
mismatch = both & pd.Series([not overlap(a, b) for a, b in zip(ours, w["wd_director"])], index=pop.index)
pop["director_mismatch"] = mismatch
print(f"Director disagrees with Wikidata: {mismatch.sum()} of {both.sum():,} films with both ({mismatch.mean():.1%} of population)")
display(pop.loc[mismatch, ["title", "year", "countries", "budget_$k", "us_gross_$k"]]
        .assign(our_director=ours[mismatch], wikidata_director=w.loc[mismatch, "wd_director"]).sort_values("budget_$k", ascending=False).head(30))

### 9m. What the Wikidata fill shows (run 2026-11-24)

| Column | Missing before | Filled | Missing after |
|---|---|---|---|
| director | 208 | 127 | 81 (1.6%) |
| writer | 266 | 136 | 130 (2.6%) |
| source_material | 891 | 158 | 733 (14.8%) |
| runtime_min | 28 | 0 | 28 (0.6%) |

**Wrong matches found:** 36 films (0.7%) have a director that disagrees with Wikidata after allowing for spelling, accents and name order. Most are real errors where our row picked up another film's data with the same title, mostly from Rotten Tomatoes: *A Christmas Carol* (2009) lists the director of a different *Christmas Carol* (it's Robert Zemeckis), *Elemental*, *Monsters vs Aliens*, *The Great Wall*, *Truth or Dare*. A few are spelling only (*Kim Gwang-sik*). **Proposed:** for these 36, use Wikidata's director, and check that their other Rotten Tomatoes columns (rating, writer, distributor) came from the right film.

**Still open:** creative type (no second source), distributor, remaining source material blanks.


### 9n. `source_group`: grouped source material, filled from TMDB keywords (agreed 2026-11-24)

Tanner's hypothesis: comic-book films vs original screenplays vs remakes is a key driver. The Numbers' 23 categories are grouped into 6: **Comic, Book, Real life, Remake, Other adaptation, Original**. Blanks are filled in order: The Numbers → Wikidata "based on" (§9l) → TMDB keywords.

Keywords are banned as a model *input* (crowd-edited after release), but here they only look up a fact that is known before production. A test on 3,801 films with both showed 74% agreement overall (Book 89%, Comic 72% but finds 93% of comic films, Original 72%, Remake 50%). `source_group_src` records where each value came from, so the model can be rerun without the keyword guesses.

In [ ]:
def kw_group(s):
    if pd.isna(s) or not str(s).strip():
        return np.nan
    t = {x.strip() for x in str(s).lower().split(",")}
    has = lambda *ks: any(k in x for x in t for k in ks)
    if has("based on comic", "superhero", "based on graphic novel", "based on manga", "comic book", "based on webcomic", "based on cartoon"):
        return "Comic"
    if has("remake", "reboot", "based on movie", "based on short film"):
        return "Remake"
    if has("based on true story", "biography", "biographical", "based on memoir", "autobiograph"):
        return "Real life"
    if has("based on novel", "based on young adult novel", "based on children's book", "based on short story", "based on magazine", "literary adaptation"):
        return "Book"
    if has("based on video game", "based on tv series", "based on play", "based on toy", "based on fairy tale", "based on myths",
           "based on theme park", "based on song", "based on anime"):
        return "Other adaptation"
    return "Original"

def tn_group(s):
    if pd.isna(s):
        return np.nan
    if "Comic" in s: return "Comic"
    if s == "Remake": return "Remake"
    if "Real Life" in s or "Factual" in s: return "Real life"
    if "Fiction Book" in s: return "Book"
    if s == "Original Screenplay": return "Original"
    return "Other adaptation"   # TV, game, play, musical, folk tale, toy, spin-off...

from_tn = cu.loc[pop.index, "source_material"].map(tn_group)        # The Numbers' own value
from_wd = pop["source_material"].map(tn_group).where(from_tn.isna())  # added by Wikidata in 9l
from_kw = pop["keywords"].map(kw_group)
pop["source_group"] = from_tn.fillna(from_wd).fillna(from_kw)
pop["source_group_src"] = np.select([from_tn.notna(), from_wd.notna(), from_kw.notna()], ["the_numbers", "wikidata", "tmdb_keywords"], "missing")

print("Where each film's source_group came from:")
display(pop["source_group_src"].value_counts().rename("films").to_frame().assign(**{"%": lambda d: (100 * d["films"] / n_pop).round(1)}))
print("source_group by origin:")
display(pd.crosstab(pop["source_group"].fillna("(missing)"), pop["source_group_src"], margins=True))

## 10. Each input vs return (training years only)

**Outcome shown:** each film's **return multiple** = US box office ÷ budget (`us_multiple`). The label (A7) cuts this at **1.25×**: at or above = winner, below = loser. Shown as a multiple, not dollars, because dollars mostly track budget (big films earn more dollars whether or not they profit). Tanner, 2026-11-24.

**Rows:** `in_population` films from **2000–2015 only** (the training years), so the 2016–19 test films stay unseen until evaluation. Data: `films_clean.csv` from the current pipeline.

**How to read a chart:** each row is a group. The dot is the group's **median** multiple; the bar spans the middle half of its films (25th–75th percentile). The dashed line is the 1.25× winner line: groups whose dot sits right of it have a typical film that's a winner. The axis is a log scale (each step doubles). Groups under 30 films are folded into "Other" or left out.

**Release year is not a model input** (Tanner): a new film's year is always later than any training year, so the model can't learn anything usable from it. It only defines the train / test / stress split.

People and companies can't be compared by name (most appear once), so §10d uses **experience**: how many earlier films the person or company had, counted from release dates (earlier films only).

In [ ]:
clean_now = pd.read_csv("../data/clean/films_clean.csv", low_memory=False)
clean_now["release_date"] = pd.to_datetime(clean_now["release_date"], format="mixed", errors="coerce")
tr = clean_now[clean_now["in_population"] & clean_now["year"].le(2015)].copy()
tr["loser"] = tr["loser"].astype(bool)
MED = tr["us_multiple"].median()
print(f"Training films: {len(tr):,}; median return {MED:.2f}x; winners (>= 1.25x) {(~tr['loser']).mean():.1%}")

SIGNAL = []   # spread between the best and worst group medians (groups with 30+ films)
TICKS = [0.05, 0.1, 0.25, 0.5, 1.25, 2, 4, 8]


def mult_table(groups, min_n=30, order=None):
    df = pd.DataFrame({"g": groups, "m": tr.loc[groups.index, "us_multiple"]})
    t = df.groupby("g", observed=True)["m"].agg(films="size", median="median",
                                                 q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75))
    t = t[t["films"] >= min_n]
    if order is not None:
        t = t.reindex([o for o in order if o in t.index])
    return t


def mult_plot(ax, groups, title, min_n=30, order=None, sort=False, record=None):
    """Median return multiple per group (dot) with the middle half of films (bar), top to bottom."""
    t = mult_table(groups, min_n, order)
    if sort:
        t = t.sort_values("median", ascending=False)
    y = np.arange(len(t))[::-1]
    ax.hlines(y, t["q25"], t["q75"], color=BLUE_LIGHT, lw=7)
    ax.plot(t["median"], y, "o", color=BLUE, ms=6)
    ax.axvline(1.25, color="#9a9890", lw=1, ls="--")
    ax.set_xscale("log"); ax.set_xlim(0.02, 10)
    ax.set_xticks(TICKS, [f"{x:g}x" for x in TICKS]); ax.minorticks_off()
    ax.set_yticks(y, [f"{i}  (n={n:,})" for i, n in zip(t.index, t["films"])])
    ax.set_xlabel("US box office / budget (median; bar = middle half)")
    ax.grid(False); ax.grid(axis="x", color="#e6e5e0", lw=0.6); ax.set_axisbelow(True)
    ax.set_title(title)
    if record:
        SIGNAL.append({"input": record, "groups": len(t),
                       "best group (median)": f"{t['median'].idxmax()} ({t['median'].max():.2f}x)",
                       "worst group (median)": f"{t['median'].idxmin()} ({t['median'].min():.2f}x)",
                       "best / worst": round(t["median"].max() / t["median"].min(), 1)})
    return t

### 10a. Money and timing: budget, runtime, release month, quarter, day of week

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
btier = pd.cut(tr["fin_budget"], [0, 5e6, 10e6, 20e6, 40e6, 80e6, np.inf], labels=["$1-5M", "$5-10M", "$10-20M", "$20-40M", "$40-80M", "$80M+"])
mult_plot(ax[0, 0], btier, "Budget tier", order=list(btier.cat.categories), record="budget")
rbin = pd.cut(tr["runtime_min"], [0, 85, 95, 105, 115, 130, np.inf], labels=["<85", "85-94", "95-104", "105-114", "115-129", "130+"], right=False)
mult_plot(ax[0, 1], rbin, "Runtime (minutes)", order=list(rbin.cat.categories), record="runtime_min")
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
mult_plot(ax[0, 2], tr["release_month"].astype(int).map(lambda m: MONTHS[m - 1]), "Release month", order=MONTHS, record="release_month")
mult_plot(ax[1, 0], tr["release_quarter"].astype(int).map(lambda q: f"Q{q}"), "Release quarter", order=["Q1", "Q2", "Q3", "Q4"], record="release_quarter")
dow = tr["release_date"].dt.day_name().where(lambda s: s.isin(["Wednesday", "Friday"]), "Other day")
mult_plot(ax[1, 1], dow, "Release day of week", order=["Wednesday", "Friday", "Other day"], record="release day of week")
ax[1, 2].axis("off")
plt.tight_layout(); plt.show()

### 10b. Content: MPAA rating, genre, source, creative type, franchise

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 11))
mult_plot(ax[0, 0], tr["mpaa_rating"].where(~tr["mpaa_rating_missing"], "blank (filled NR)"), "MPAA rating",
          order=["G", "PG", "PG-13", "R", "NC-17", "NR", "blank (filled NR)"], record="mpaa_rating")
gen = tr["genres_all"].str.split(",").explode().str.strip().dropna()
mult_plot(ax[0, 1], gen, "Genre (a film counts in each of its genres)", sort=True, record="genres_all")
mult_plot(ax[1, 0], tr["source_group"].fillna("(blank)"), "Source material group (Tanner's hypothesis)", sort=True, record="source_group")
mult_plot(ax[1, 1], tr["creative_type"].fillna("(blank)"), "Creative type", sort=True, record="creative_type")
plt.tight_layout(); plt.show()

In [ ]:
# Franchise: as tagged, and the version known before release (an earlier film in the same franchise was already out)
fr = tr["franchise"].ne("(none)")
allf = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna() & clean_now["franchise"].ne("(none)")][["franchise", "release_date"]]
first = allf.assign(f=allf["franchise"].str.split("|")).explode("f").groupby("f")["release_date"].min()
prior = pd.Series([fs != "(none)" and any(first.get(f, pd.NaT) < d for f in str(fs).split("|"))
                   for fs, d in zip(tr["franchise"], tr["release_date"])], index=tr.index)
known = pd.Series(np.select([prior, fr], ["sequel / later film", "first film of a franchise"], "no franchise"), index=tr.index)
fig, ax = plt.subplots(1, 2, figsize=(15, 3.2))
mult_plot(ax[0], fr.map({True: "tagged franchise", False: "no franchise"}), "Franchise (as tagged; tags first films in hindsight)")
mult_plot(ax[1], known, "Franchise, known before release", order=["no franchise", "first film of a franchise", "sequel / later film"],
          record="franchise (earlier film exists)")
plt.tight_layout(); plt.show()

### 10c. Origin: country and language

In [ ]:
us = tr["countries"].fillna("")
cgroup = np.select([us.eq("United States of America"), us.str.contains("United States")], ["US only", "US co-production"], "non-US")
fig, ax = plt.subplots(1, 2, figsize=(15, 3.2))
mult_plot(ax[0], pd.Series(cgroup, index=tr.index), "Production country", order=["US only", "US co-production", "non-US"], record="countries (US group)")
mult_plot(ax[1], tr["original_language"].eq("en").map({True: "English", False: "not English"}), "Original language", record="original_language")
plt.tight_layout(); plt.show()

### 10d. People and companies: experience from earlier films

For each training film, how many **earlier** released films (any year from 2000, in the cleaned data) its director, writer or production company already had. Names alone are too thin: most appear once.

In [ ]:
hist = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna()][["film_uid", "release_date", "director", "writer", "production_companies", "distributor"]]


def prior_count(col, top_only=False):
    """For each training film: the most earlier films any one of its listed people/companies had."""
    h = hist[["film_uid", "release_date", col]].dropna()
    h = h.assign(name=h[col].str.split(",")).explode("name")
    h["name"] = h["name"].str.strip()
    h = h.sort_values("release_date")
    h["prior"] = h.groupby("name").cumcount()          # films by this name released earlier
    best = h.groupby("film_uid")["prior"].max()
    return tr["film_uid"].map(best)


EXP = ["0 (first film)", "1", "2-4", "5-9", "10+"]
exp_bins = lambda s: pd.cut(s, [-1, 0, 1, 4, 9, np.inf], labels=EXP)
fig, ax = plt.subplots(1, 3, figsize=(17, 3.6))
for a, col in zip(ax, ["director", "writer", "production_companies"]):
    mult_plot(a, exp_bins(prior_count(col)).astype(object).fillna("(blank)"), f"{col}: earlier films",
              order=EXP + ["(blank)"], record=f"{col} experience")
plt.tight_layout(); plt.show()

In [ ]:
# Distributor: the biggest names vs everyone else (names are not standardized yet, §6h)
d = tr["distributor"].fillna("(blank)").str.split(",").str[0].str.strip()
top = d.value_counts().head(14).index
fig, ax = plt.subplots(figsize=(9, 6))
mult_plot(ax, d.where(d.isin(top), "Other"), "Distributor (first listed; top 14 by films)", sort=True, record="distributor")
plt.tight_layout(); plt.show()

### 10e. Do the "missing" flags carry signal?

Blanks aren't random (they cluster in small films), so a blank may itself predict the outcome.

In [ ]:
rows = []
for c in [c for c in tr.columns if c.endswith("_missing")]:
    blank = tr[c].astype(bool)
    if blank.sum() >= 30:
        mb, mf = tr.loc[blank, "us_multiple"].median(), tr.loc[~blank, "us_multiple"].median()
        rows.append({"input": c.removesuffix("_missing"), "films blank": int(blank.sum()),
                     "median return if blank": f"{mb:.2f}x", "median return if filled": f"{mf:.2f}x", "filled / blank": round(mf / mb, 1)})
display(pd.DataFrame(rows).sort_values("filled / blank", ascending=False).set_index("input"))

### 10f. Which inputs separate returns most?

Best / worst = the best group's median return divided by the worst group's (groups with 30+ films). A rough first ranking only: it ignores group size and overlap between inputs.

In [ ]:
display(pd.DataFrame(SIGNAL).sort_values("best / worst", ascending=False).set_index("input"))

### 10g. What the return comparisons show (run 2026-11-24, training years only)

Overall: 3,708 training films. **Median return 0.72×** (US box office ÷ budget); 33.1% reach the 1.25× winner line. Numbers below are group medians.

**Strong signals**
1. **MPAA rating:** G 1.31× → PG 1.08× → PG-13 0.96× → R 0.56× → **NR 0.05×**. 80% of NR films are non-US; NR mostly means a foreign or limited release.
2. **Origin:** US-only 1.02×, US co-productions 0.70×, **non-US 0.11×**. Language says the same (English 0.85×, not English 0.12×). Under a US box-office outcome, foreign films rarely earn back their budget in the US.
3. **Budget (clearer than with the winner rate):** films under $10M have a median of only 0.24–0.27×, vs 0.78–0.99× above $10M. Small films usually don't earn their budget back in US theaters, though their spread is the widest (a few break out).
4. **Sequels:** films with an earlier franchise film already released: 1.45×, vs 0.54× for non-franchise films.
5. **Genre:** Family 1.17×, Documentary 1.05×, Animation 1.00× at the top; Drama 0.50×, History 0.36×, Western and War 0.32× at the bottom.
6. **Creative type:** Super Hero 1.20×, Kids Fiction 1.18× at the top; Dramatization 0.57×, Historical Fiction 0.54× at the bottom.
7. **Distributor:** Warner Bros. 1.20× at the top, IFC Films 0.02× at the bottom (names not yet standardized).
8. **Experience:** directors on their first film 0.57× → 10+ earlier films 1.07×. Production companies with 10+ earlier films 0.93× vs 0.17–0.50× for less experienced ones.

**Tanner's source hypothesis: supported on returns.** Other adaptation (TV, games, toys) 1.06×, Comic 0.96×, Remake 0.95× vs Book 0.78×, Real life 0.67×, **Original 0.67×**. Adapted, known properties return roughly 40% more than original screenplays. (The winner-rate view hid this: the groups differ in how much they make, more than in how often they cross 1.25×.)

**Weak signals:** runtime (0.54× for 130+ min up to 0.92× under 85 min), release month (Sep–Oct and Apr–May 0.55–0.63×; Jun, Dec, Jul 0.82–0.84×), Wednesday openings 1.04× vs Friday 0.66×, quarter (0.69–0.81×).

**Not a model input: release year** (Tanner): a new film's year is always later than any training year.

**⚠️ Two warnings before these become model inputs**
- **Blank films return almost nothing (median 0.03–0.14×).** A blank usually means the databases never documented the film, and they document successful films better. So a blank partly records the outcome. The `_missing` flags (and "blank" groups) may leak it; test the model with and without them.
- **"First film of a franchise" is hindsight.** Those films return 1.72×, because a film only becomes the first of a franchise *after* it succeeds and gets sequels. Only "an earlier film was already released" (sequel) is known before release (matches Matt's A8 condition).
